# Estimacion LA-AIDS -- Demanda de Alimentos (ENAHO)

Notebook reestructurado para que sea facil de correr de principio a fin:

- **Parte 1 (construccion de la base de datos)**: son los pasos mas
  pesados (leen archivos .dta / parquet grandes, procesan por lotes).
  Cada celda **se salta automaticamente** si su archivo de salida ya
  existe en el disco, para no repetir trabajo cada vez que corres el
  notebook. Si cambias algo upstream (el diccionario de productos, el
  mapeo de grupos, etc.) y necesitas reconstruir todo desde cero, pon
  `FORZAR_RECONSTRUCCION = True` en la celda de configuracion de abajo
  y vuelve a correr.

- **Parte 2 (regresiones y simulacion)**: son rapidas (segundos), y es
  la parte que normalmente se re-corre al iterar (agregar variables,
  probar submuestras, etc.). Estas celdas **siempre se ejecutan**, no
  tienen checkpoint.

- **Apendice**: diagnosticos y exploraciones que ya cumplieron su
  proposito (llevaron a decisiones de diseno que ya estan incorporadas
  en el pipeline final) o que son opcionales para investigar problemas
  puntuales. No se corren automaticamente al ejecutar todo el notebook
  de arriba a abajo -- se dejan documentadas por si se necesitan de
  nuevo.

**Como usar:** corre las celdas en orden, de arriba a abajo (Kernel ->
Restart & Run All, o Cell -> Run All). La primera vez tardara varios
minutos (arma toda la base). Las siguientes veces, la Parte 1 se salta
casi instantaneamente y solo corren las regresiones de la Parte 2.

## Parte 0: Configuracion

Todos los scripts de este notebook asumen que los archivos de datos
(`.dta`, `.parquet`, `.csv`) estan en el **mismo directorio** desde el
que corres Jupyter (revisa con `%pwd` si no estas seguro).

In [2]:
import os

# Pon esto en True si quieres FORZAR que se reconstruya toda la base de
# datos desde cero (Parte 1), aunque los archivos de salida ya existan.
# Uso tipico: cambiaste el diccionario de productos, el mapeo de grupos,
# o alguna regla de construccion, y necesitas que se recalcule todo.
FORZAR_RECONSTRUCCION = False

print("Directorio de trabajo actual:", os.getcwd())
print(f"FORZAR_RECONSTRUCCION = {FORZAR_RECONSTRUCCION}")


Directorio de trabajo actual: c:\Users\Usuario\Documents\Est demanda alimentos
FORZAR_RECONSTRUCCION = False


## Parte 1: Construccion de la base de datos

Pasos pesados (leen los microdatos crudos de la ENAHO y arman las
bases intermedias). **Cada celda se salta si su(s) archivo(s) de
salida ya existen** -- ver Parte 0. La UNICA excepcion es el Paso 1
(instalar paquetes), que siempre se ejecuta -- ver por que abajo.

Orden de dependencias (cada paso lee la salida del anterior):

1. Instala TODOS los paquetes que el resto del notebook necesita (pandas,
   numpy, pyarrow, statsmodels, linearmodels, psutil).
2. `enaho01-2025-601.dta` --> `enaho_601_filtrado.parquet` (filtra canasta basica + comida rapida)
3. `enaho_601_filtrado.parquet` --> `enaho_601_sin_missings.parquet` (quita missings en compra)
4. `enaho_601_sin_missings.parquet` --> `enaho_601_renombrado.parquet` (columnas legibles)
5. `enaho_601_renombrado.parquet` --> `enaho_601_con_wi.parquet` (participacion de gasto w_i por producto)
6. `diccionario_produc61.csv` --> `mapeo_grupo_aids.csv` (clasifica productos en categorias tematicas)
7. `mapeo_grupo_aids.csv` --> `mapeo_grupo_aids_v2.csv` (subdivide pan y pollo por sesgo de calidad)
8. `enaho01-2025-200.dta`, `enaho01a-2025-500.dta` --> `enaho_200.parquet`, `enaho_500.parquet`
9. `enaho_200.parquet` + `enaho_500.parquet` --> `controles_hogar.parquet` (tamano hogar, jefe/a, ingreso total hogar)
10. `enaho_601_con_wi.parquet` + `mapeo_grupo_aids_v2.csv` + `controles_hogar.parquet` + `enaho_200/500.parquet`
    --> **`base_laids_final.parquet`** (base final: precios ajustados por calidad de Deaton, indice de Stone,
    instrumentos leave-one-out, e ingreso del jefe/a de hogar empleado/obrero) -- **esta es la base que usa
    toda la Parte 2.**

### Paso 1 -- Instalar todos los paquetes necesarios

Esta celda **siempre se ejecuta primero y sin checkpoint**: revisa si
cada paquete ya esta instalado en el kernel actual (`importlib.import_module`)
e instala solo lo que falte. Va antes que cualquier otro paso porque
incluso leer/escribir un archivo parquet (Paso 2 en adelante) requiere
`pyarrow`. Es necesaria en cualquier kernel/entorno nuevo (p.ej. si
cambias de interprete de Python en VS Code), aunque ya hayas generado
los archivos intermedios con otro entorno antes.

**Si esta celda instala algo nuevo, te lo va a decir en rojo/negrita al
final: REINICIA EL KERNEL y vuelve a correr el notebook desde el
principio.** Esto no es opcional para paquetes como `pyarrow` (tienen
extensiones compiladas en C) -- instalarlos y usarlos en el mismo
proceso de Python sin reiniciar puede producir errores raros tipo
`ArrowKeyError: type extension ... already defined`, porque el kernel
ya tiene cargada una version parcial en memoria. Si ya estaban todos
instalados, no hace falta reiniciar nada y puedes seguir corriendo
normalmente.

In [3]:
import importlib
import subprocess
import sys

# nombre del modulo a importar -> nombre del paquete en PyPI (a veces difieren)
PAQUETES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "pyarrow": "pyarrow",       # necesario para TODO el I/O de parquet (pd.read_parquet / to_parquet)
    "statsmodels": "statsmodels",
    "linearmodels": "linearmodels",
    "psutil": "psutil",         # usado para reportar RAM disponible
}

_paquetes_recien_instalados = []

for _modulo, _paquete in PAQUETES.items():
    try:
        importlib.import_module(_modulo)
        print(f"'{_paquete}' ya esta instalado -- OK.")
    except ImportError:
        print(f"Instalando '{_paquete}' (no encontrado en este kernel: {sys.executable})...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", _paquete])
        print(f"'{_paquete}' instalado -- OK.")
        _paquetes_recien_instalados.append(_paquete)

if _paquetes_recien_instalados:
    print("\n" + "!" * 78)
    print(f"SE ACABAN DE INSTALAR (por primera vez en este kernel): {_paquetes_recien_instalados}")
    print("ACCION REQUERIDA: reinicia el kernel (Kernel -> Restart) y vuelve a correr")
    print("el notebook desde el principio. Paquetes con extensiones compiladas (como")
    print("pyarrow) pueden fallar con errores tipo 'ArrowKeyError: type extension ya")
    print("definida' si se usan en el mismo proceso donde se acaban de instalar --")
    print("el kernel necesita arrancar de nuevo para cargarlos limpio.")
    print("!" * 78)
else:
    print("\nTodos los paquetes necesarios ya estaban instalados -- puedes seguir corriendo normalmente.")

# Uso posterior (referencia rapida):
#   OLS  ->  import statsmodels.api as sm ; sm.OLS(y, sm.add_constant(X)).fit()
#   IV   ->  from linearmodels.iv import IV2SLS
#   SUR  ->  from linearmodels.system import SUR / IV3SLS


'pandas' ya esta instalado -- OK.
'numpy' ya esta instalado -- OK.
Instalando 'pyarrow' (no encontrado en este kernel: c:\Users\Usuario\AppData\Local\Python\pythoncore-3.14-64\python.exe)...
'pyarrow' instalado -- OK.
'statsmodels' ya esta instalado -- OK.
'linearmodels' ya esta instalado -- OK.
'psutil' ya esta instalado -- OK.

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
SE ACABAN DE INSTALAR (por primera vez en este kernel): ['pyarrow']
ACCION REQUERIDA: reinicia el kernel (Kernel -> Restart) y vuelve a correr
el notebook desde el principio. Paquetes con extensiones compiladas (como
pyarrow) pueden fallar con errores tipo 'ArrowKeyError: type extension ya
definida' si se usan en el mismo proceso donde se acaban de instalar --
el kernel necesita arrancar de nuevo para cargarlos limpio.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


### Paso 2 -- Convertir modulo 601 a Parquet y filtrar canasta basica + comida rapida

In [4]:
# --- paso2_filtrar: se salta si ya existe la salida (ver FORZAR_RECONSTRUCCION) ---
_SALIDAS_paso2_filtrar = ['enaho_601_filtrado.parquet']

def _ejecutar_paso2_filtrar():
    """
    convertir_y_filtrar.py
    ------------------------
    1. Convierte enaho01-2025-601.dta a Parquet (por chunks, para no saturar la RAM)
    2. Filtra los productos de CANASTA BASICA y COMIDA RAPIDA segun el codigo (601)
    3. Guarda un parquet final ya filtrado, listo para trabajar

    IMPORTANTE: revisa la seccion "CONFIGURACION" mas abajo antes de correr,
    porque el nombre exacto de la columna de codigo puede variar segun tu base.
    """

    import pandas as pd
    import os
    import gc
    import struct

    # ============================================================
    # DIAGNOSTICO RAPIDO (revisa esto primero en la consola)
    # ============================================================
    print(f"Python de {struct.calcsize('P') * 8} bits")
    try:
        import psutil
        mem = psutil.virtual_memory()
        print(f"RAM total: {mem.total / 1e9:.1f} GB | RAM disponible: {mem.available / 1e9:.1f} GB")
    except ImportError:
        print("(instala 'psutil' con pip para ver el detalle de RAM disponible)")

    # ============================================================
    # CONFIGURACION -- AJUSTA ESTO SEGUN TU BASE
    # ============================================================
    ARCHIVO_DTA = "enaho01-2025-601.dta"
    CARPETA_PARQUET = "parquet_partes"          # carpeta temporal con las partes
    ARCHIVO_PARQUET_FILTRADO = "enaho_601_filtrado.parquet"
    TAMANO_CHUNK = 20_000                       # bajado bastante; sube o baja segun tu RAM

    # Diccionario ya validado de codigo -> clasificacion (canasta_basica / comida_rapida / otro)
    DICCIONARIO_CSV = "diccionario_produc61.csv"

    # Si solo necesitas ciertas columnas, listalas aqui para reducir memoria
    # desde la lectura misma (deja en None para leer todas).
    COLUMNAS_A_LEER = None

    # Nombre de la columna que contiene el codigo de producto (601).
    COLUMNA_CODIGO = "produc61"

    # ============================================================
    # 1. CARGAR CODIGOS A CONSERVAR DESDE EL DICCIONARIO VALIDADO
    # ============================================================
    def cargar_codigos_a_conservar():
        dicc = pd.read_csv(DICCIONARIO_CSV)
        codigos = dicc.loc[
            dicc["clasificacion"].isin(["canasta_basica", "comida_rapida"]),
            "codigo"
        ]
        codigos_set = set(codigos.astype(float))
        print(f"Codigos a conservar (canasta basica + comida rapida): {len(codigos_set)}")
        return codigos_set


    CODIGOS_A_CONSERVAR = cargar_codigos_a_conservar()

    # ============================================================
    # 2. CONVERTIR DTA A PARQUET POR PARTES
    # ============================================================
    def convertir_a_parquet():
        os.makedirs(CARPETA_PARQUET, exist_ok=True)
        print(f"Convirtiendo {ARCHIVO_DTA} a Parquet en partes de {TAMANO_CHUNK} filas...\n")

        partes = []
        lector = pd.read_stata(
            ARCHIVO_DTA,
            chunksize=TAMANO_CHUNK,
            columns=COLUMNAS_A_LEER,
            convert_categoricals=False,  # evita error por etiquetas duplicadas (ej. p601b3)
        )
        for i, chunk in enumerate(lector):
            ruta_parte = os.path.join(CARPETA_PARQUET, f"parte_{i:03d}.parquet")
            chunk.to_parquet(ruta_parte, index=False)
            partes.append(ruta_parte)
            print(f"  Parte {i} guardada ({len(chunk)} filas) -> {ruta_parte}")

            # liberar memoria explicitamente antes de leer el siguiente chunk
            del chunk
            gc.collect()

        print(f"\nConversion completa. {len(partes)} partes generadas.")
        return partes


    def unir_partes(partes):
        """
        NOTA: si tu RAM es muy limitada, esta funcion puede volver a fallar
        porque intenta juntar TODO en memoria. Si te da MemoryError aqui,
        usa directamente filtrar_por_partes() en vez de unir_partes()
        + filtrar_productos() -- ve la seccion EJECUCION mas abajo.
        """
        print("\nUniendo partes en un solo archivo parquet...")
        df_completo = pd.concat(
            [pd.read_parquet(p) for p in partes],
            ignore_index=True
        )
        df_completo.to_parquet(ARCHIVO_PARQUET_COMPLETO, index=False)
        print(f"Archivo completo guardado en: {ARCHIVO_PARQUET_COMPLETO}")
        print(f"Filas totales: {len(df_completo):,}")
        return df_completo


    # ============================================================
    # 3. FILTRAR CANASTA BASICA + COMIDA RAPIDA
    # ============================================================
    def filtrar_por_partes(partes):
        """
        Version que NUNCA carga la base completa en memoria.
        Filtra cada parte por separado usando el diccionario validado
        de codigos (CODIGOS_A_CONSERVAR), y va acumulando solo las
        filas que sobreviven el filtro.
        """
        piezas_filtradas = []
        total_antes = 0

        for i, ruta in enumerate(partes):
            chunk = pd.read_parquet(ruta)
            total_antes += len(chunk)

            if COLUMNA_CODIGO not in chunk.columns:
                print(f"\n[ATENCION] No se encontro la columna '{COLUMNA_CODIGO}'.")
                print("Columnas disponibles:")
                print(list(chunk.columns))
                raise ValueError(
                    "Ajusta la variable COLUMNA_CODIGO al inicio del script."
                )

            chunk_filtrado = chunk[chunk[COLUMNA_CODIGO].isin(CODIGOS_A_CONSERVAR)]
            piezas_filtradas.append(chunk_filtrado)

            print(f"  Parte {i}: {len(chunk)} filas -> {len(chunk_filtrado)} filas filtradas")

            del chunk, chunk_filtrado
            gc.collect()

        df_filtrado = pd.concat(piezas_filtradas, ignore_index=True)
        df_filtrado.to_parquet(ARCHIVO_PARQUET_FILTRADO, index=False)

        print(f"\nFilas totales antes de filtrar: {total_antes:,}")
        print(f"Filas totales despues de filtrar: {len(df_filtrado):,}")
        print(f"Base filtrada guardada en: {ARCHIVO_PARQUET_FILTRADO}")
        return df_filtrado


    # ============================================================
    # EJECUCION
    # ============================================================
    if __name__ == "__main__":
        import glob

        # Si ya corriste la conversion antes (ya existen las partes en disco),
        # no la repitas: solo recolecta las rutas ya guardadas.
        partes_existentes = sorted(glob.glob(os.path.join(CARPETA_PARQUET, "parte_*.parquet")))

        if partes_existentes:
            print(f"Se encontraron {len(partes_existentes)} partes ya convertidas. Saltando conversion.")
            partes = partes_existentes
        else:
            partes = convertir_a_parquet()

        df_filtrado = filtrar_por_partes(partes)

        print("\n=== LISTO ===")
        print(df_filtrado.head())


if FORZAR_RECONSTRUCCION or not all(os.path.exists(_a) for _a in _SALIDAS_paso2_filtrar):
    _ejecutar_paso2_filtrar()
else:
    print(f"[SALTADO] Ya existen: {_SALIDAS_paso2_filtrar} -- no se reconstruye este paso.")
    print("(Pon FORZAR_RECONSTRUCCION = True en la celda de configuracion si quieres forzar el rebuild.)")


[SALTADO] Ya existen: ['enaho_601_filtrado.parquet'] -- no se reconstruye este paso.
(Pon FORZAR_RECONSTRUCCION = True en la celda de configuracion si quieres forzar el rebuild.)


### Paso 3 -- Quitar missings en la variable de compra (`p601a1`)

In [5]:
# --- paso3_missings: se salta si ya existe la salida (ver FORZAR_RECONSTRUCCION) ---
_SALIDAS_paso3_missings = ['enaho_601_sin_missings.parquet']

def _ejecutar_paso3_missings():
    """
    quitar_missings.py (version por lotes, para RAM limitada)
    ------------------------------------------------------------
    Lee enaho_601_filtrado.parquet en lotes pequenos usando PyArrow,
    quita las filas con missing en 'p601a1', y escribe el resultado
    directamente a un nuevo parquet SIN cargar la base completa en RAM.
    """

    import pyarrow.parquet as pq
    import pyarrow as pa

    ARCHIVO_ENTRADA = "enaho_601_filtrado.parquet"
    ARCHIVO_SALIDA = "enaho_601_sin_missings.parquet"
    COLUMNA = "p601a1"
    TAMANO_LOTE = 50_000  # filas por lote; bajar si sigue fallando

    archivo_pq = pq.ParquetFile(ARCHIVO_ENTRADA)

    total_antes = 0
    total_despues = 0
    escritor = None

    try:
        for lote in archivo_pq.iter_batches(batch_size=TAMANO_LOTE):
            tabla = pa.Table.from_batches([lote])
            total_antes += tabla.num_rows

            # Filtrar filas donde la columna NO es null
            mascara = tabla[COLUMNA].is_valid()
            tabla_filtrada = tabla.filter(mascara)
            total_despues += tabla_filtrada.num_rows

            if escritor is None:
                escritor = pq.ParquetWriter(ARCHIVO_SALIDA, tabla_filtrada.schema)

            escritor.write_table(tabla_filtrada)

            print(f"Lote procesado: {tabla.num_rows} filas -> {tabla_filtrada.num_rows} sin missing")

    finally:
        if escritor is not None:
            escritor.close()

    print(f"\nFilas totales antes: {total_antes:,}")
    print(f"Filas totales despues (sin missing en '{COLUMNA}'): {total_despues:,}")
    print(f"Base guardada en: {ARCHIVO_SALIDA}")


if FORZAR_RECONSTRUCCION or not all(os.path.exists(_a) for _a in _SALIDAS_paso3_missings):
    _ejecutar_paso3_missings()
else:
    print(f"[SALTADO] Ya existen: {_SALIDAS_paso3_missings} -- no se reconstruye este paso.")
    print("(Pon FORZAR_RECONSTRUCCION = True en la celda de configuracion si quieres forzar el rebuild.)")


[SALTADO] Ya existen: ['enaho_601_sin_missings.parquet'] -- no se reconstruye este paso.
(Pon FORZAR_RECONSTRUCCION = True en la celda de configuracion si quieres forzar el rebuild.)


### Paso 4 -- Renombrar columnas a nombres mas legibles

In [6]:
# --- paso4_renombrar: se salta si ya existe la salida (ver FORZAR_RECONSTRUCCION) ---
_SALIDAS_paso4_renombrar = ['enaho_601_renombrado.parquet']

def _ejecutar_paso4_renombrar():
    import pandas as pd

    # ============================================================
    # 2. RENOMBRAR COLUMNAS A NOMBRES MAS LEGIBLES
    # ============================================================
    ARCHIVO_ENTRADA = "enaho_601_sin_missings.parquet"
    ARCHIVO_SALIDA = "enaho_601_renombrado.parquet"

    # Mapeo CONFIRMADO con las etiquetas oficiales del INEI (variable_labels del .dta):
    RENOMBRE = {
        # Identificadores del hogar / encuesta
        "conglome":   "conglomerado",
        "vivienda":   "vivienda",
        "hogar":      "hogar",
        "produc61":   "codigo_producto",     # codigo interno CCIF (el que usamos para filtrar)
        "p601a":      "codigo_producto_601", # "Codigo del producto" segun el modulo 601 (similar/redundante a produc61)
        "p601x":      "nombre_producto",
        "p601b":      "obtuvo_producto",     # pregunta general: obtuvo/consumio/compro/le regalaron (no es dato de compra)

        # 601-A: como obtuvo el producto (flags, uno por alternativa)
        "p601a1": "obtuvo_comprado",
        "p601a2": "obtuvo_autoconsumo",
        "p601a3": "obtuvo_autosuministro",
        "p601a4": "obtuvo_pago_a_miembro_hogar",
        "p601a5": "obtuvo_regalo_otro_hogar",
        "p601a6": "obtuvo_programa_social",
        "p601a7": "obtuvo_otro",

        # 601-B: datos de la COMPRA ("Con que frecuencia compro / cantidad de compra / donde")
        "p601b1": "frecuencia_compra",
        "p601b2": "cantidad_comprada",
        "p601b3": "unidad_medida_compra",
        "p601b4": "lugar_compra",

        # 601-C: monto gastado en la compra
        "p601c":  "gasto",

        # 601-D: frecuencia/cantidad OBTENIDA-consumida en general (no exclusivo de compra)
        "p601d1": "frecuencia_consumo",
        "p601d2": "cantidad_consumida",
        "p601d3": "unidad_medida_consumo",

        # Variables IMPUTADAS y DEFLACTADAS/ANUALIZADAS del INEI (utiles para regresiones,
        # ya vienen con missings imputados por el propio INEI)
        "i601b2": "cantidad_comprada_kg_imputada_anual",
        "i601c":  "gasto_imputado_anual",
        "d601c":  "gasto_deflactado_anual",
        "i601d2": "cantidad_consumida_kg_imputada_anual",
        "i601e":  "monto_estimado_imputado_anual",

        # Otro (bienes libres / no libres) - poco usado, se deja documentado
        "t601a71": "obtuvo_otro_sin_bienes_libres",
        "t601a72": "obtuvo_otro_bienes_libres",
    }

    print("\nCargando base para renombrar (por lotes, para no saturar RAM)...")

    import pyarrow.parquet as pq
    import pyarrow as pa

    archivo_pq = pq.ParquetFile(ARCHIVO_ENTRADA)
    escritor = None
    total = 0

    try:
        for lote in archivo_pq.iter_batches(batch_size=50_000):
            tabla = pa.Table.from_batches([lote])

            # Renombrar solo las columnas que existen y estan en el mapeo
            nuevos_nombres = [RENOMBRE.get(c, c) for c in tabla.column_names]
            tabla = tabla.rename_columns(nuevos_nombres)

            if escritor is None:
                escritor = pq.ParquetWriter(ARCHIVO_SALIDA, tabla.schema)
            escritor.write_table(tabla)
            total += tabla.num_rows
            print(f"  Lote procesado: {tabla.num_rows} filas")
    finally:
        if escritor is not None:
            escritor.close()

    print(f"\nTotal de filas procesadas: {total:,}")
    print(f"Base renombrada guardada en: {ARCHIVO_SALIDA}")

    # Vista previa
    muestra = pd.read_parquet(ARCHIVO_SALIDA, engine="pyarrow").head(3) if total < 100_000 else None
    if muestra is None:
        # leer solo un lote pequeno para vista previa, sin cargar todo
        primer_lote = next(pq.ParquetFile(ARCHIVO_SALIDA).iter_batches(batch_size=5))
        muestra = pa.Table.from_batches([primer_lote]).to_pandas()

    print("\nColumnas finales:")
    print(list(muestra.columns))
    print("\nVista previa:")
    print(muestra)


if FORZAR_RECONSTRUCCION or not all(os.path.exists(_a) for _a in _SALIDAS_paso4_renombrar):
    _ejecutar_paso4_renombrar()
else:
    print(f"[SALTADO] Ya existen: {_SALIDAS_paso4_renombrar} -- no se reconstruye este paso.")
    print("(Pon FORZAR_RECONSTRUCCION = True en la celda de configuracion si quieres forzar el rebuild.)")


[SALTADO] Ya existen: ['enaho_601_renombrado.parquet'] -- no se reconstruye este paso.
(Pon FORZAR_RECONSTRUCCION = True en la celda de configuracion si quieres forzar el rebuild.)


### Paso 5 -- Calcular participacion de gasto `w_i` por producto y hogar

In [7]:
# --- paso5_wi: se salta si ya existe la salida (ver FORZAR_RECONSTRUCCION) ---
_SALIDAS_paso5_wi = ['enaho_601_con_wi.parquet']

def _ejecutar_paso5_wi():
    """
    calcular_wi.py
    -----------------
    Calcula, para cada hogar:
      gasto_total_i = suma de 'gasto' de todos los productos j del hogar i
    Y luego, para cada fila (producto j del hogar i):
      w_ij = gasto_ij / gasto_total_i

    Se hace en DOS PASADAS por lotes (sin cargar toda la base en RAM):
      Pasada 1: acumula el gasto total por hogar en un diccionario.
      Pasada 2: recorre de nuevo y calcula w_ij, escribiendo el resultado.

    El identificador de HOGAR se arma con: conglomerado + vivienda + hogar
    (el Modulo 601 esta a nivel de hogar, no existe identificador de persona).
    """

    import pyarrow.parquet as pq
    import pyarrow as pa
    import pandas as pd

    ARCHIVO_ENTRADA = "enaho_601_renombrado.parquet"
    ARCHIVO_SALIDA = "enaho_601_con_wi.parquet"

    # Columnas que identifican el hogar (ajusta si tus nombres son distintos)
    COLUMNAS_ID_HOGAR = ["conglomerado", "vivienda", "hogar"]
    COLUMNA_GASTO = "gasto"

    TAMANO_LOTE = 50_000


    def construir_llave_hogar(df):
        """Concatena las columnas identificadoras en una sola llave (string)."""
        return df[COLUMNAS_ID_HOGAR].astype(str).agg("_".join, axis=1)


    # ============================================================
    # PASADA 1: acumular gasto_total por hogar
    # ============================================================
    print("=== PASADA 1: calculando gasto total por hogar ===")

    archivo_pq = pq.ParquetFile(ARCHIVO_ENTRADA)
    gasto_total_por_hogar = {}  # llave_hogar -> suma acumulada de gasto

    for i, lote in enumerate(archivo_pq.iter_batches(batch_size=TAMANO_LOTE)):
        df = pa.Table.from_batches([lote]).to_pandas()
        df["_llave_hogar"] = construir_llave_hogar(df)

        suma_lote = df.groupby("_llave_hogar")[COLUMNA_GASTO].sum(min_count=1)

        for llave, suma in suma_lote.items():
            if pd.isna(suma):
                continue
            gasto_total_por_hogar[llave] = gasto_total_por_hogar.get(llave, 0) + suma

        if i % 20 == 0:
            print(f"  Lote {i} procesado (pasada 1)...")

    print(f"Total de hogares distintos encontrados: {len(gasto_total_por_hogar):,}")

    # ============================================================
    # PASADA 2: calcular w_ij y escribir resultado
    # ============================================================
    print("\n=== PASADA 2: calculando w_ij y escribiendo resultado ===")

    escritor = None
    total_filas = 0

    for i, lote in enumerate(archivo_pq.iter_batches(batch_size=TAMANO_LOTE)):
        df = pa.Table.from_batches([lote]).to_pandas()
        df["_llave_hogar"] = construir_llave_hogar(df)

        df["gasto_total_hogar"] = df["_llave_hogar"].map(gasto_total_por_hogar)
        df["w_i"] = df[COLUMNA_GASTO] / df["gasto_total_hogar"]

        df = df.drop(columns="_llave_hogar")

        tabla_salida = pa.Table.from_pandas(df, preserve_index=False)

        if escritor is None:
            escritor = pq.ParquetWriter(ARCHIVO_SALIDA, tabla_salida.schema)
        escritor.write_table(tabla_salida)

        total_filas += len(df)
        if i % 20 == 0:
            print(f"  Lote {i} procesado (pasada 2)...")

    if escritor is not None:
        escritor.close()

    print(f"\nTotal de filas procesadas: {total_filas:,}")
    print(f"Base con 'gasto_total_hogar' y 'w_i' guardada en: {ARCHIVO_SALIDA}")

    # Vista previa rapida (solo un lote pequeno, sin cargar todo)
    primer_lote = next(pq.ParquetFile(ARCHIVO_SALIDA).iter_batches(batch_size=10))
    muestra = pa.Table.from_batches([primer_lote]).to_pandas()
    print("\nVista previa:")
    print(muestra[COLUMNAS_ID_HOGAR + [COLUMNA_GASTO, "gasto_total_hogar", "w_i"]])


if FORZAR_RECONSTRUCCION or not all(os.path.exists(_a) for _a in _SALIDAS_paso5_wi):
    _ejecutar_paso5_wi()
else:
    print(f"[SALTADO] Ya existen: {_SALIDAS_paso5_wi} -- no se reconstruye este paso.")
    print("(Pon FORZAR_RECONSTRUCCION = True en la celda de configuracion si quieres forzar el rebuild.)")


[SALTADO] Ya existen: ['enaho_601_con_wi.parquet'] -- no se reconstruye este paso.
(Pon FORZAR_RECONSTRUCCION = True en la celda de configuracion si quieres forzar el rebuild.)


### Paso 6 -- Clasificar productos en categorias tematicas de alimentos (grupos AIDS)

In [8]:
# --- paso6_mapeo: se salta si ya existe la salida (ver FORZAR_RECONSTRUCCION) ---
_SALIDAS_paso6_mapeo = ['mapeo_grupo_aids.csv']

def _ejecutar_paso6_mapeo():
    """
    reclasificar_grupos_aids.py (v2 - reglas ampliadas)
    ------------------------------------------------------
    Clasifica cada producto en categorias tematicas de alimentos
    (las que usa la literatura de sistemas AIDS), usando palabras
    clave sobre la descripcion real, normalizando tildes y con
    patrones flexibles a plurales/variantes.

    Genera: mapeo_grupo_aids.csv  (codigo, descripcion, grupo_aids)
    """

    import pandas as pd
    import re
    import unicodedata

    ARCHIVO_DICCIONARIO = "diccionario_produc61.csv"
    SALIDA = "mapeo_grupo_aids.csv"

    dicc = pd.read_csv(ARCHIVO_DICCIONARIO)


    def quitar_tildes(texto):
        texto = unicodedata.normalize("NFKD", str(texto))
        return "".join(c for c in texto if not unicodedata.combining(c))


    # ============================================================
    # Reglas de clasificacion (orden importa: la primera que matchee gana)
    # ============================================================
    REGLAS = [
        ("comida_fuera_hogar", [
            r"pollo a la brasa", r"chifa", r"parrillada", r"salchipapa",
            r"anticucho", r"restaurante", r"\bmenu\b(?!dencia)", r"pizza",
            r"tamal", r"cevich", r"desayuno en", r"cena en", r"lonche",
            r"comedor", r"pension", r"servid[oa]",
            r"alimentos al paso", r"almuerzo en kiosko", r"chicharron",
            r"cuartel", r"comidas en rest", r"ensalada de frutas", r"humita",
            r"juanes", r"pachamanca", r"pollada", r"sandwich", r"\bcausa\b",
            r"chilcano", r"cocktel", r"coctel", r"wantan", r"\bsopa", r"\bpure",
        ]),
        ("bebidas_alcoholicas", [
            r"cerveza", r"whisky", r"\bron\b", r"pisco", r"\bvino\b",
            r"aguardiente", r"anisado", r"brandy", r"cachina", r"licor",
            r"chicha de jora", r"champagne", r"cog.ac", r"\bgin\b", r"sidra",
            r"vermouth", r"vodka", r"guarapo", r"\bmasato\b", r"chicha de mani",
            r"bebidas alcoholicas",
        ]),
        ("bebidas_no_alcoholicas", [
            r"gaseosa", r"agua mineral", r"agua de mesa", r"nectar",
            r"\bjugo", r"rehidratante", r"energizante", r"^agua de",
            r"aguajina", r"chicha de frutas", r"chicha morada", r"emoliente",
            r"\bmalta\b", r"maltina", r"refresco", r"\bhumari\b", r"^life\b",
        ]),
        ("cafe_te_infusiones", [
            r"\bcafe\b", r"\bte\b", r"cocoa", r"hierba", r"infusion",
            r"achicoria", r"^anis\b", r"canchalagua", r"cedron", r"mate\b",
            r"hierbabuena", r"manzanilla", r"cascarrilla",
            r"\bboldo\b", r"chuchuhuasi", r"\bcoca\b", r"llipza", r"\bculen\b",
            r"eucalipto", r"grama dulce", r"huamanripa", r"llanten", r"\bmalva\b",
            r"\bmatico\b", r"\bmenta\b", r"\bmolle\b", r"\bmuna\b", r"\bortiga\b",
            r"\bretama\b", r"\bromero\b", r"\bruda\b", r"\bsalvia\b", r"\bsabila\b",
            r"\btilo\b", r"toronjil", r"valeriana", r"\bverbena\b", r"papaina",
            r"\baltea\b", r"\bpanti\b", r"cola de caballo", r"\bcacao\b", r"camu.?camu", r"chanca piedra", r"una de gato",
        ]),
        ("azucar_dulces", [
            r"azucar", r"caramelo", r"chocolate", r"\bmiel\b", r"mermelada",
            r"galleta.*dulce", r"gelatina", r"\bflan\b", r"mazamorra", r"pastel",
            r"biscotela", r"chantilly", r"algarrobina", r"bombon", r"chancaca",
            r"confite", r"crema volteada", r"chisito", r"papitas fritas",
            r"\bchifle\b", r"\bchup", r"endulzante", r"gomas? de mascar", r"champus",
            r"\bhelado", r"jalea real", r"king kong", r"manjar blanco",
            r"\bmelaza\b", r"otras golosinas", r"\bpapilla", r"\bpostre",
            r"\bpudin\b", r"\btoffe", r"turron",
        ]),
        ("pan_pasteleria_cereales", [
            r"\bpan\b", r"^pan\s", r"\bpanes?\b", r"bizcocho", r"biscocho", r"\btorta",
            r"empanada", r"galleta", r"harina", r"fideo", r"\bavena",
            r"\btrigo", r"quinua", r"maicena", r"semola", r"\barroz", r"\bmaiz",
            r"\bcancha\b", r"chochoca", r"\bmote\b", r"kiwicha", r"cereal",
            r"canigua", r"\bcebada\b", r"\bcenteno\b", r"macarron", r"apanar",
            r"reposteria", r"panaderia", r"\bpasta", r"\bpolenta\b",
            r"polvo de hornear", r"\blevadura\b", r"ravioles", r"tallarin",
            r"\btostada", r"paneton", r"\bqueque", r"maicillo", r"bicarbonato",
        ]),
        ("frutos_secos_semillas", [
            r"almendra", r"castana", r"\bmani\b", r"\bnuez\b", r"\bnueces\b",
            r"pecana", r"ajonjoli", r"linaza",
        ]),
        ("carnes_rojas", [
            r"\bres\b", r"carnero", r"cordero", r"chancho", r"cerdo", r"alpaca",
            r"\bcuy\b", r"higado", r"mondongo", r"\bbofe\b", r"pata de res",
            r"hueso de res", r"tocino", r"chorizo", r"jamon", r"hot dog",
            r"hamburguesa", r"cabrito", r"caprino", r"carne\b",
            r"cabeza o pata de llama", r"\bchalona\b", r"\bcharqui\b",
            r"\bcecina\b", r"criadilla", r"embutido", r"mortadela", r"salchicha",
            r"otras carnes\b", r"otras carnes procesadas", r"\bpellejo\b",
            r"\brinon\b", r"viscera", r"sangre de", r"\bconejo\b", r"\bllama\b",
            r"\bmono\b", r"\bronsoco\b", r"\bsajino\b", r"\brana\b", r"animales vivos",
        ]),
        ("pollo_aves", [
            r"pollo", r"gallina", r"\bpavo\b", r"\bpato\b", r"^ave\b", r"menudencia",
            r"\bpaloma\b", r"otras carnes de aves",
        ]),
        ("pescados_mariscos", [
            r"pescado", r"jurel", r"bonito", r"caballa", r"merluza",
            r"cojinova", r"\bperico\b", r"\btoyo\b", r"\bliza\b", r"pejerrey",
            r"boquichico", r"atun", r"sardina", r"\bchoro", r"cangrejo",
            r"camaron", r"\bpota\b", r"marisco", r"calamar", r"pulpo", r"concha",
            r"\bbagre\b", r"bacalao", r"albacora", r"\baguja\b", r"cabinza",
            r"cabrilla", r"cachema", r"carachama", r"carachi", r"\bchita\b",
            r"chiu-chiu", r"\bcongrio\b", r"corvina", r"doncella", r"\bdorado\b",
            r"\bguitarra\b", r"langosta", r"langostino", r"lenguado", r"\blorna\b",
            r"\bmachas\b", r"mejillon", r"\bmero\b", r"mojarrilla", r"palometa",
            r"\bpaco\b", r"paiche", r"\bpeje\b", r"pintadilla", r"\brobalo\b",
            r"salmon", r"sabalo", r"\bsierra\b", r"\bsuche\b", r"tilapia",
            r"\btollo\b", r"trucha", r"tucumare", r"gamitana", r"zungaro",
            r"\braya\b", r"\bbatea\b", r"\bmanta\b", r"caracol", r"\berizo\b",
            r"muy-muy", r"\balga", r"conchita", r"maruchas", r"almeja", r"yuyos", r"chanco de mar", r"corvinilla",
        ]),
        ("lacteos_huevos", [
            r"\bleche", r"queso", r"yogurt", r"yogourt", r"mantequilla", r"margarina",
            r"\bhuevo", r"crema de leche", r"cuajad", r"quesillo", r"reque[sz]on",
        ]),
        ("aceites_grasas", [
            r"\baceite", r"\bmanteca", r"\bsebo\b",
        ]),
        ("verduras_hortalizas", [
            r"\bpapa\b", r"papas\b", r"camote", r"\byuca\b", r"olluco", r"cebolla",
            r"tomate", r"zanahoria", r"zapallo", r"choclo", r"\bapio\b",
            r"\bajo\b", r"lechuga", r"culantro", r"\bporo\b", r"beterraga",
            r"pepinillo", r"brocoli", r"\baji\b", r"especia", r"pimienta",
            r"canela", r"clavo de olor", r"oregano", r"comino", r"\bsal\b",
            r"chuno", r"nabo", r"rocoto", r"paico", r"perejil",
            r"\bacelga\b", r"achiote", r"alcachofa", r"berenjena", r"\bberros\b",
            r"\bcaigua\b", r"calabaza", r"\bcol\b", r"coliflor", r"dale dale",
            r"escorsonera", r"esparrago", r"espinaca", r"\bhinojo\b", r"huacatay",
            r"\bizano\b", r"\bmashua\b", r"\boca\b", r"tuberculo", r"raices",
            r"\bpepino\b", r"pimenton", r"pimiento", r"\bpituca\b", r"rabanito",
            r"\bvainita\b", r"verdura", r"\byacon\b", r"\bpalillo\b", r"curcuma",
            r"azafran", r"mostaza", r"ketchup", r"mayonesa", r"sazonador",
            r"sillao", r"sibarita", r"nuez moscada", r"\btara\b", r"\bocopa\b",
            r"condimento", r"colorante", r"legumbres", r"\bvinagre\b", r"albahaca", r"cubito", r"\bkion\b", r"jengibre", r"\bmaca\b", r"\bhongos\b", r"\blaurel\b",
        ]),
        ("frutas", [
            r"limon", r"naranja", r"mandarina", r"papaya", r"platano",
            r"manzana", r"^pina\b", r"\buva\b", r"melon", r"sandia", r"palta",
            r"aceituna", r"fresa", r"mango", r"granadilla", r"chirimoya",
            r"maracuya", r"lucuma", r"tuna\b",
            r"albaricoque", r"arandano", r"caimito", r"camu camu", r"capuli",
            r"\bcereza\b", r"\bciruela\b", r"\bcoco\b", r"cocona", r"damasco",
            r"durazno", r"\bdatil", r"\bfrutilla\b", r"\bgranada\b", r"guaba",
            r"\bpacae\b", r"guanabana", r"guayaba", r"\bguinda", r"\bhigo\b",
            r"\blima\b", r"\bmamey\b", r"maranon", r"membrillo", r"\bmoras\b",
            r"\bnispero\b", r"\borejon", r"\bpasas\b", r"\bpera\b", r"\bpijuayo\b",
            r"pomarrosa", r"tamarindo", r"taperiba", r"toronja", r"pomelo",
            r"\btumbo\b", r"\buvas\b", r"uvillas", r"\bzapote\b", r"confitada", r"guindon", r"huesillo", r"otras frutas",
        ]),
        ("menestras", [
            r"lenteja", r"frijol", r"frejol", r"arveja", r"arvejon", r"\bhaba",
            r"pallar", r"garbanzo", r"soya\b", r"lentejon", r"chocho\b",
            r"tarhui", r"habitas", r"alverja", r"\bmenestra",
        ]),
        ("alimento_mascotas_animales", [
            r"mascota", r"\bperro", r"\bgatos?\b", r"alfalfa", r"afrecho",
            r"nicovita", r"conejita", r"alpiste", r"manayupa", r"happy dog",
            r"alimento.*animales", r"residuos vegetales",
        ]),
    ]


    def clasificar(desc):
        d = quitar_tildes(str(desc).lower())
        for nombre_grupo, patrones in REGLAS:
            if any(re.search(p, d) for p in patrones):
                return nombre_grupo
        return "otros_sin_clasificar"


    dicc["grupo_aids"] = dicc["descripcion"].apply(clasificar)
    dicc.to_csv(SALIDA, index=False, encoding="utf-8-sig")

    print("=== RESUMEN DE GRUPOS AIDS ===\n")
    tabla_resumen = dicc.groupby("grupo_aids").agg(
        n_productos=("codigo", "count"),
        ejemplos=("descripcion", lambda x: ", ".join(x.head(4)))
    ).sort_values("n_productos", ascending=False)

    pd.set_option("display.max_colwidth", 150)
    pd.set_option("display.width", 200)
    print(tabla_resumen.to_string())
    print(f"\nMapeo completo guardado en: {SALIDA}")

    sin_clasificar = dicc.loc[dicc["grupo_aids"] == "otros_sin_clasificar", "descripcion"].sort_values()
    sin_clasificar.to_csv("productos_sin_clasificar.csv", index=False, encoding="utf-8-sig")

    print(f"\n=== TODOS los {len(sin_clasificar)} productos sin clasificar (residual esperado) ===")
    for desc in sin_clasificar:
        print(f"  - {desc}")


if FORZAR_RECONSTRUCCION or not all(os.path.exists(_a) for _a in _SALIDAS_paso6_mapeo):
    _ejecutar_paso6_mapeo()
else:
    print(f"[SALTADO] Ya existen: {_SALIDAS_paso6_mapeo} -- no se reconstruye este paso.")
    print("(Pon FORZAR_RECONSTRUCCION = True en la celda de configuracion si quieres forzar el rebuild.)")


[SALTADO] Ya existen: ['mapeo_grupo_aids.csv'] -- no se reconstruye este paso.
(Pon FORZAR_RECONSTRUCCION = True en la celda de configuracion si quieres forzar el rebuild.)


### Paso 7 -- Subdividir categorias heterogeneas (pan y pollo)

*(Motivo, ver Apendice "Diagnostico de signos raros": pan_pasteleria_cereales
y pollo_aves mostraban sesgo de calidad -- precio pagado correlacionado con
el gasto total del hogar -- que se corrige subdividiendolos en componentes
mas homogeneos.)*

In [9]:
# --- paso7_subdividir: se salta si ya existe la salida (ver FORZAR_RECONSTRUCCION) ---
_SALIDAS_paso7_subdividir = ['mapeo_grupo_aids_v2.csv']

def _ejecutar_paso7_subdividir():
    """
    subdividir_grupos_heterogeneos.py
    -------------------------------------
    Subdivide las 2 categorias que mostraron sesgo de calidad
    (correlacion alta entre precio pagado y gasto total del hogar):

      pan_pasteleria_cereales  ->  pan_basico  vs  pasteleria_dulce
      pollo_aves                ->  pollo_piezas  vs  menudencia_aves

    Genera un mapeo actualizado: mapeo_grupo_aids_v2.csv
    """

    import pandas as pd
    import re
    import unicodedata

    ARCHIVO_MAPEO = "mapeo_grupo_aids.csv"
    SALIDA = "mapeo_grupo_aids_v2.csv"


    def quitar_tildes(texto):
        texto = unicodedata.normalize("NFKD", str(texto))
        return "".join(c for c in texto if not unicodedata.combining(c))


    mapeo = pd.read_csv(ARCHIVO_MAPEO)

    # ============================================================
    # Reglas para subdividir PAN_PASTELERIA_CEREALES
    # ============================================================
    PALABRAS_PASTELERIA_DULCE = [
        r"bizcocho", r"biscocho", r"\btorta", r"galleta.*dulce", r"galleta$",
        r"paneton", r"queque", r"pastel", r"biscotela", r"chantilly",
        r"turron", r"empanada",
    ]
    # todo lo demas del grupo original (pan corriente, arroz, fideos, harinas,
    # avena, quinua, cereales, etc.) se queda como "pan_basico"

    def subdividir_pan(desc):
        d = quitar_tildes(str(desc).lower())
        if any(re.search(p, d) for p in PALABRAS_PASTELERIA_DULCE):
            return "pasteleria_dulce"
        return "pan_basico"


    # ============================================================
    # Reglas para subdividir POLLO_AVES
    # ============================================================
    PALABRAS_MENUDENCIA_AVES = [r"menudencia"]

    def subdividir_pollo(desc):
        d = quitar_tildes(str(desc).lower())
        if any(re.search(p, d) for p in PALABRAS_MENUDENCIA_AVES):
            return "menudencia_aves"
        return "pollo_piezas"


    # ============================================================
    # Aplicar subdivision
    # ============================================================
    mapeo["grupo_aids_v2"] = mapeo["grupo_aids"]

    es_pan = mapeo["grupo_aids"] == "pan_pasteleria_cereales"
    mapeo.loc[es_pan, "grupo_aids_v2"] = mapeo.loc[es_pan, "descripcion"].apply(subdividir_pan)

    es_pollo = mapeo["grupo_aids"] == "pollo_aves"
    mapeo.loc[es_pollo, "grupo_aids_v2"] = mapeo.loc[es_pollo, "descripcion"].apply(subdividir_pollo)

    mapeo.to_csv(SALIDA, index=False, encoding="utf-8-sig")

    print("=== RESUMEN grupo_aids_v2 (con subdivision) ===\n")
    resumen = mapeo.groupby("grupo_aids_v2").agg(
        n_productos=("codigo", "count"),
        ejemplos=("descripcion", lambda x: ", ".join(x.head(4)))
    ).sort_values("n_productos", ascending=False)

    pd.set_option("display.max_colwidth", 150)
    pd.set_option("display.width", 200)
    print(resumen.to_string())

    print(f"\nMapeo actualizado guardado en: {SALIDA}")
    print("\nGrupos nuevos creados:")
    print("  - pan_basico (pan corriente, arroz, fideos, harinas, cereales, quinua, etc.)")
    print("  - pasteleria_dulce (biscochos, tortas, galletas dulces, panetones, queques)")
    print("  - pollo_piezas (pollo entero, pechuga, pierna, gallina, pavo, pato)")
    print("  - menudencia_aves (menudencias de pollo, gallina, pato, pavo)")


if FORZAR_RECONSTRUCCION or not all(os.path.exists(_a) for _a in _SALIDAS_paso7_subdividir):
    _ejecutar_paso7_subdividir()
else:
    print(f"[SALTADO] Ya existen: {_SALIDAS_paso7_subdividir} -- no se reconstruye este paso.")
    print("(Pon FORZAR_RECONSTRUCCION = True en la celda de configuracion si quieres forzar el rebuild.)")


[SALTADO] Ya existen: ['mapeo_grupo_aids_v2.csv'] -- no se reconstruye este paso.
(Pon FORZAR_RECONSTRUCCION = True en la celda de configuracion si quieres forzar el rebuild.)


### Paso 8 -- Leer modulos 200 (composicion del hogar) y 500 (empleo e ingresos)

In [10]:
# --- paso8_leer_200_500: se salta si ya existe la salida (ver FORZAR_RECONSTRUCCION) ---
_SALIDAS_paso8_leer_200_500 = ['enaho_200.parquet', 'enaho_500.parquet']

def _ejecutar_paso8_leer_200_500():
    """
    leer_modulos_200_500.py
    ---------------------------
    Lee los modulos 200 (Caracteristicas de los miembros del hogar) y
    500 (Empleo e Ingresos) de la ENAHO, los convierte a Parquet, y
    extrae las etiquetas de variable reales del INEI para identificar
    con certeza las columnas que necesitamos para el ajuste de calidad
    de Deaton (tamaño del hogar, educacion, area urbano/rural, ingreso).
    """

    import pandas as pd
    import psutil

    ARCHIVO_200 = "enaho01-2025-200.dta"
    ARCHIVO_500 = "enaho01a-2025-500.dta"

    SALIDA_200 = "enaho_200.parquet"
    SALIDA_500 = "enaho_500.parquet"

    mem = psutil.virtual_memory()
    print(f"RAM disponible: {mem.available / 1e9:.1f} GB\n")

    # ============================================================
    # 1. Modulo 200 - Caracteristicas de los miembros del hogar
    # ============================================================
    print("=== Leyendo modulo 200 (Caracteristicas del hogar) ===")

    reader_200 = pd.io.stata.StataReader(ARCHIVO_200)
    etiquetas_200 = reader_200.variable_labels()

    df_200 = pd.read_stata(ARCHIVO_200, convert_categoricals=False)
    print(f"Filas: {len(df_200):,}  |  Columnas: {len(df_200.columns)}")
    df_200.to_parquet(SALIDA_200, index=False)
    print(f"Guardado en: {SALIDA_200}\n")

    print("--- Etiquetas de columnas relevantes (parentesco, sexo, edad, educacion, area) ---")
    prefijos_relevantes = ("p203", "p207", "p208", "p301", "p300", "estrato", "area", "ubigeo",
                            "conglome", "vivienda", "hogar", "codperso", "dominio", "mes")
    for col in df_200.columns:
        if col.lower().startswith(prefijos_relevantes):
            etiqueta = etiquetas_200.get(col, "(sin etiqueta)")
            print(f"  {col:15s} -> {etiqueta}")

    # ============================================================
    # 2. Modulo 500 - Empleo e Ingresos
    # ============================================================
    print("\n\n=== Leyendo modulo 500 (Empleo e Ingresos) ===")

    reader_500 = pd.io.stata.StataReader(ARCHIVO_500)
    etiquetas_500 = reader_500.variable_labels()

    df_500 = pd.read_stata(ARCHIVO_500, convert_categoricals=False)
    print(f"Filas: {len(df_500):,}  |  Columnas: {len(df_500.columns)}")
    df_500.to_parquet(SALIDA_500, index=False)
    print(f"Guardado en: {SALIDA_500}\n")

    print("--- Etiquetas de columnas relevantes (ingreso, ocupacion) ---")
    prefijos_ingreso = ("i524", "i530", "i538", "i541", "i544", "i556", "i557", "i558",
                         "p507", "p510", "p513", "conglome", "vivienda", "hogar", "codperso")
    for col in df_500.columns:
        if col.lower().startswith(prefijos_ingreso):
            etiqueta = etiquetas_500.get(col, "(sin etiqueta)")
            print(f"  {col:15s} -> {etiqueta}")

    print("\n=== LISTO ===")
    print("Revisa las etiquetas de arriba para confirmar los nombres exactos de:")
    print("  - tamaño del hogar (se puede derivar contando personas por conglome+vivienda+hogar)")
    print("  - nivel educativo (probablemente en el modulo 300, no en 200 -- avisa si lo necesitas)")
    print("  - area urbano/rural (probablemente 'estrato' o similar)")
    print("  - ingreso total del hogar (probablemente hay que sumar varias columnas de ingreso)")


if FORZAR_RECONSTRUCCION or not all(os.path.exists(_a) for _a in _SALIDAS_paso8_leer_200_500):
    _ejecutar_paso8_leer_200_500()
else:
    print(f"[SALTADO] Ya existen: {_SALIDAS_paso8_leer_200_500} -- no se reconstruye este paso.")
    print("(Pon FORZAR_RECONSTRUCCION = True en la celda de configuracion si quieres forzar el rebuild.)")


[SALTADO] Ya existen: ['enaho_200.parquet', 'enaho_500.parquet'] -- no se reconstruye este paso.
(Pon FORZAR_RECONSTRUCCION = True en la celda de configuracion si quieres forzar el rebuild.)


### Paso 9 -- Construir controles a nivel de hogar (tamano, jefe/a, ingreso total)

In [11]:
# --- paso9_controles: se salta si ya existe la salida (ver FORZAR_RECONSTRUCCION) ---
_SALIDAS_paso9_controles = ['controles_hogar.parquet']

def _ejecutar_paso9_controles():
    """
    construir_controles_hogar.py
    --------------------------------
    Construye variables de control a nivel de HOGAR a partir de los
    modulos 200 (Caracteristicas del hogar) y 500 (Empleo e Ingresos),
    para usarlas en el ajuste de calidad de Deaton (1988):

      - tamano_hogar: numero de miembros del hogar
      - jefe_sexo, jefe_edad: caracteristicas del jefe/a de hogar
      - estrato: estrato geografico (proxy de area/tamaño de ciudad)
      - ingreso_total_hogar: suma de todos los ingresos monetarios
        reportados por los miembros del hogar (imputados por INEI)

    Salida: controles_hogar.parquet (uno por hogar)
    """

    import pandas as pd
    import numpy as np

    ARCHIVO_200 = "enaho_200.parquet"
    ARCHIVO_500 = "enaho_500.parquet"
    SALIDA = "controles_hogar.parquet"

    COLUMNAS_ID = ["conglome", "vivienda", "hogar"]

    # ============================================================
    # 1. Modulo 200: tamaño del hogar y caracteristicas del jefe
    # ============================================================
    print("=== Procesando modulo 200 ===")
    df_200 = pd.read_parquet(ARCHIVO_200)

    # Tamaño del hogar: cuenta de personas por hogar
    tamano_hogar = df_200.groupby(COLUMNAS_ID).size().rename("tamano_hogar")

    # Caracteristicas del jefe/a de hogar (p203 == 1)
    jefes = df_200[df_200["p203"] == 1].copy()
    jefes = jefes.drop_duplicates(subset=COLUMNAS_ID)  # por si hay duplicados raros
    jefes = jefes.set_index(COLUMNAS_ID)[["p207", "p208a", "estrato", "dominio", "mes"]]
    jefes = jefes.rename(columns={"p207": "jefe_sexo", "p208a": "jefe_edad"})

    controles_200 = pd.concat([tamano_hogar, jefes], axis=1).reset_index()
    print(f"Hogares en modulo 200: {len(controles_200):,}")

    # ============================================================
    # 2. Modulo 500: ingreso total del hogar
    # ============================================================
    print("\n=== Procesando modulo 500 ===")
    df_500 = pd.read_parquet(ARCHIVO_500)

    COLUMNAS_INGRESO = [
        "i524a1",  # ingreso total, ocupacion principal, trabajo dependiente
        "i530a",   # ganancia neta, ocupacion principal, trabajo independiente
        "i538a1",  # ingreso total, ocupacion secundaria, trabajo dependiente
        "i541a",   # ganancia neta, ocupacion secundaria, trabajo independiente
    ]
    columnas_presentes = [c for c in COLUMNAS_INGRESO if c in df_500.columns]
    print(f"Columnas de ingreso encontradas: {columnas_presentes}")

    df_500["_ingreso_persona"] = df_500[columnas_presentes].apply(
        lambda fila: fila.fillna(0).sum(), axis=1
    )

    ingreso_hogar = df_500.groupby(COLUMNAS_ID)["_ingreso_persona"].sum().rename("ingreso_total_hogar")
    controles_500 = ingreso_hogar.reset_index()
    print(f"Hogares en modulo 500: {len(controles_500):,}")

    # ============================================================
    # 3. Unir todo a nivel de hogar
    # ============================================================
    print("\n=== Uniendo controles ===")
    controles = controles_200.merge(controles_500, on=COLUMNAS_ID, how="left")

    # el ingreso puede quedar NaN si nadie en el hogar tiene 14+ años o no trabaja
    controles["ingreso_total_hogar"] = controles["ingreso_total_hogar"].fillna(0)

    controles = controles.rename(columns={"conglome": "conglomerado"})
    for col in ["conglomerado", "vivienda", "hogar"]:
        controles[col] = controles[col].astype(str)

    controles.to_parquet(SALIDA, index=False)
    print(f"\nControles guardados en: {SALIDA}  ({len(controles):,} hogares)")

    # ============================================================
    # Estadisticas descriptivas rapidas
    # ============================================================
    print("\n=== Estadisticas descriptivas ===")
    print(controles[["tamano_hogar", "jefe_edad", "ingreso_total_hogar"]].describe())

    print("\nDistribucion jefe_sexo (1=Hombre, 2=Mujer segun codificacion estandar ENAHO):")
    print(controles["jefe_sexo"].value_counts())

    print("\nVista previa:")
    print(controles.head())


if FORZAR_RECONSTRUCCION or not all(os.path.exists(_a) for _a in _SALIDAS_paso9_controles):
    _ejecutar_paso9_controles()
else:
    print(f"[SALTADO] Ya existen: {_SALIDAS_paso9_controles} -- no se reconstruye este paso.")
    print("(Pon FORZAR_RECONSTRUCCION = True en la celda de configuracion si quieres forzar el rebuild.)")


[SALTADO] Ya existen: ['controles_hogar.parquet'] -- no se reconstruye este paso.
(Pon FORZAR_RECONSTRUCCION = True en la celda de configuracion si quieres forzar el rebuild.)


### Paso 10 -- Construir la base FINAL del LA-AIDS (`base_laids_final.parquet`)

Version final: 17 grupos (pan y pollo ya subdivididos), precios
ajustados por calidad (Deaton 1988), indice de Stone household-specific,
instrumentos leave-one-out, e **ingreso individual del jefe/a de hogar**
(solo empleado/obrero, seccion 3B) para poder estimar el efecto del
ingreso y simular cambios en la RMV en la Parte 2.

In [12]:
# --- paso10_base_final: se salta si ya existe la salida (ver FORZAR_RECONSTRUCCION) ---
_SALIDAS_paso10_base_final = ['base_laids_final.parquet']

def _ejecutar_paso10_base_final():
    """
    construir_base_laids_final.py
    ---------------------------------
    Version FINAL de la construccion de la base para el LA-AIDS:

      1. Usa el mapeo SUBDIVIDIDO (mapeo_grupo_aids_v2.csv): 17 grupos
         -- separa pan_pasteleria_cereales en pan_basico / pasteleria_dulce,
            y pollo_aves en pollo_piezas / menudencia_aves.

      2. Aplica el AJUSTE DE CALIDAD DE DEATON (1988) sobre el precio de
         cada bien: para los hogares que SI compraron (precio real), se
         regresiona ln(precio) contra caracteristicas DEMOGRAFICAS del
         hogar (tamaño del hogar, edad y sexo del jefe/a) y se 'purga' la
         parte del precio explicada por tener caracteristicas distintas
         al hogar promedio. Esto separa la eleccion de calidad
         (correlacionada con caracteristicas del hogar) del precio de
         mercado real.

         IMPORTANTE (Deaton 1988): el gasto total del hogar (ln_X) NO se
         incluye entre las variables de purga Z. ln_X es la base del
         indice de Stone y por tanto de ln(X/P*), el regresor de interes
         de todo el sistema AIDS. Si se usara ln_X para purgar el precio,
         el precio "ajustado" de cada hogar quedaria correlacionado por
         construccion con su propio gasto -- una relacion mecanica, no
         economica -- y esa correlacion se trasladaria a ln(X/P*) al
         recalcular el indice de Stone con esos precios. El resultado es
         una endogeneidad artificial que contamina justo el coeficiente
         que el sistema busca estimar (beta_i, elasticidad-gasto). Por
         eso Z contiene solo caracteristicas demograficas/composicion del
         hogar, nunca el gasto ni el ingreso total.

      3. Recalcula el instrumento leave-one-out y el indice de Stone
         usando los precios YA AJUSTADOS.

    Salida: base_laids_final.parquet
    """

    import pyarrow.parquet as pq
    import pyarrow as pa
    import pandas as pd
    import numpy as np
    import statsmodels.api as sm

    ARCHIVO_ENTRADA = "enaho_601_con_wi.parquet"
    ARCHIVO_MAPEO = "mapeo_grupo_aids_v2.csv"
    ARCHIVO_CONTROLES = "controles_hogar.parquet"
    SALIDA = "base_laids_final.parquet"

    COLUMNAS_ID_HOGAR = ["conglomerado", "vivienda", "hogar"]
    COLUMNA_CODIGO = "codigo_producto"
    COLUMNA_GASTO = "gasto"
    COLUMNA_CANTIDAD = "cantidad_comprada"
    COLUMNA_DOMINIO = "dominio"
    COLUMNA_MES = "mes"

    GRUPOS_EXCLUIDOS = ["alimento_mascotas_animales", "otros_sin_clasificar"]
    TAMANO_LOTE = 50_000


    def llave_hogar(df):
        return df[COLUMNAS_ID_HOGAR].astype(str).agg("_".join, axis=1)


    # ============================================================
    # 1. Mapeo codigo -> grupo (version subdividida)
    # ============================================================
    mapeo = pd.read_csv(ARCHIVO_MAPEO)
    mapeo["codigo"] = mapeo["codigo"].astype(int)
    mapa_grupo = dict(zip(mapeo["codigo"], mapeo["grupo_aids_v2"]))

    GRUPOS = sorted(g for g in mapeo["grupo_aids_v2"].unique() if g not in GRUPOS_EXCLUIDOS)
    print(f"Grupos a usar en el sistema ({len(GRUPOS)}): {GRUPOS}")

    # ============================================================
    # 2. Pasada por lotes: acumular gasto/cantidad por hogar x grupo
    # ============================================================
    print("\n=== Acumulando gasto y cantidad por hogar x grupo ===")

    archivo_pq = pq.ParquetFile(ARCHIVO_ENTRADA)
    gasto_acum, cantidad_acum, info_hogar = {}, {}, {}

    for i, lote in enumerate(archivo_pq.iter_batches(batch_size=TAMANO_LOTE)):
        df = pa.Table.from_batches([lote]).to_pandas()
        df["_llave_hogar"] = llave_hogar(df)
        df["_grupo"] = df[COLUMNA_CODIGO].map(mapa_grupo)
        df = df[df["_grupo"].isin(GRUPOS)]
        df_validos = df.dropna(subset=["_grupo", COLUMNA_GASTO])

        suma_gasto = df_validos.groupby(["_llave_hogar", "_grupo"])[COLUMNA_GASTO].sum(min_count=1)
        suma_cantidad = df_validos.groupby(["_llave_hogar", "_grupo"])[COLUMNA_CANTIDAD].sum(min_count=1)

        for llave, valor in suma_gasto.items():
            if pd.isna(valor):
                continue
            gasto_acum[llave] = gasto_acum.get(llave, 0) + valor
        for llave, valor in suma_cantidad.items():
            if pd.isna(valor):
                continue
            cantidad_acum[llave] = cantidad_acum.get(llave, 0) + valor

        info_lote = df.drop_duplicates("_llave_hogar")[["_llave_hogar", COLUMNA_DOMINIO, COLUMNA_MES]]
        for _, fila in info_lote.iterrows():
            if fila["_llave_hogar"] not in info_hogar:
                info_hogar[fila["_llave_hogar"]] = (fila[COLUMNA_DOMINIO], fila[COLUMNA_MES])

        if i % 20 == 0:
            print(f"  Lote {i} procesado...")

    print(f"Combinaciones hogar x grupo CON compra: {len(gasto_acum):,}")

    # ============================================================
    # 3. Tabla de compras + tabla de hogares (con controles del 200/500)
    # ============================================================
    filas = []
    for (llave, grupo), gasto_i in gasto_acum.items():
        cantidad_i = cantidad_acum.get((llave, grupo), np.nan)
        filas.append({"_llave_hogar": llave, "grupo": grupo, "gasto_i": gasto_i, "cantidad_i": cantidad_i})

    compras = pd.DataFrame(filas)
    compras["precio_i"] = compras["gasto_i"] / compras["cantidad_i"]
    compras.loc[compras["cantidad_i"] <= 0, "precio_i"] = np.nan
    compras["ln_precio_i"] = np.where(compras["precio_i"] > 0, np.log(compras["precio_i"]), np.nan)

    X_hogar = compras.groupby("_llave_hogar")["gasto_i"].sum().rename("X")

    hogares = pd.DataFrame(
        [(k, v[0], v[1]) for k, v in info_hogar.items()],
        columns=["_llave_hogar", "dominio", "mes"]
    )
    hogares = hogares.merge(X_hogar, on="_llave_hogar", how="left").dropna(subset=["X"])
    hogares["ln_X"] = np.log(hogares["X"])

    print(f"\nHogares con al menos una compra: {len(hogares):,}")

    print("\n=== Uniendo controles del hogar (modulos 200 y 500) ===")
    controles = pd.read_parquet(ARCHIVO_CONTROLES)
    controles["_llave_hogar"] = controles[COLUMNAS_ID_HOGAR].astype(str).agg("_".join, axis=1)

    controles_reducido = controles[["_llave_hogar", "tamano_hogar", "jefe_edad", "jefe_sexo"]]

    hogares = hogares.merge(controles_reducido, on="_llave_hogar", how="left")
    n_sin_control = hogares[["tamano_hogar", "jefe_edad", "jefe_sexo"]].isna().any(axis=1).sum()
    print(f"Hogares sin datos de control (no cruzaron con modulo 200/500): {n_sin_control:,} de {len(hogares):,}")

    # ============================================================
    # 3B. Ingreso PRINCIPAL del jefe/a de hogar, restringido a
    #     empleado/obrero (P507 = 3 o 4), desde los modulos crudos de la
    #     ENAHO:
    #       - enaho_200.parquet: parentesco (P203 = 1 -> jefe/a de hogar)
    #       - enaho_500.parquet: categoria ocupacional (P507), frecuencia
    #         de pago (P523) e ingreso en esa frecuencia (P524A1,
    #         "ingreso total" de la ocupacion principal por trabajo
    #         dependiente)
    #
    #     P530 (ganancia neta del trabajador independiente) NO aplica
    #     aqui: esa pregunta solo se le hace a P507 = 2 (trabajador
    #     independiente), nunca a empleado/obrero -- por eso no se usa.
    #
    #     P524A1 se convierte a un monto MENSUAL segun la frecuencia de
    #     pago declarada en P523 (1=diario, 2=semanal, 3=quincenal,
    #     4=mensual), usando factores estandar (30 dias/mes, 30/7
    #     semanas/mes, 2 quincenas/mes). Es una aproximacion razonable,
    #     no exacta (no usa dias/horas realmente trabajados).
    # ============================================================
    print("\n=== Ingreso del jefe/a de hogar (empleado/obrero) desde modulos 200 y 500 crudos ===")

    ARCHIVO_MODULO_200_PERSONA = "enaho_200.parquet"
    ARCHIVO_MODULO_500_PERSONA = "enaho_500.parquet"
    COLUMNAS_ID_PERSONA = ["conglome", "vivienda", "hogar", "codperso"]
    CODIGO_JEFE_HOGAR = 1.0          # P203 == 1 -> jefe/a de hogar
    CODIGOS_EMPLEADO_OBRERO = [3.0, 4.0]  # P507: 3 = empleado, 4 = obrero
    FACTOR_MENSUAL = {1.0: 30.0, 2.0: 30.0 / 7.0, 3.0: 2.0, 4.0: 1.0}  # segun P523

    mod200 = pd.read_parquet(ARCHIVO_MODULO_200_PERSONA,
                              columns=COLUMNAS_ID_PERSONA + ["p203"])
    jefes = mod200.loc[mod200["p203"] == CODIGO_JEFE_HOGAR, COLUMNAS_ID_PERSONA].copy()
    print(f"Jefes/as de hogar identificados (P203=1): {len(jefes):,}")

    mod500 = pd.read_parquet(ARCHIVO_MODULO_500_PERSONA,
                              columns=COLUMNAS_ID_PERSONA + ["p507", "p523", "p524a1"])

    ingreso_jefe = jefes.merge(mod500, on=COLUMNAS_ID_PERSONA, how="left")
    n_sin_500 = ingreso_jefe["p507"].isna().sum()
    print(f"Jefes/as sin fila en modulo 500 (no se les aplico la seccion de empleo): {n_sin_500:,}")

    ingreso_jefe = ingreso_jefe[ingreso_jefe["p507"].isin(CODIGOS_EMPLEADO_OBRERO)].copy()
    print(f"Jefes/as empleado/obrero (P507 in {CODIGOS_EMPLEADO_OBRERO}): {len(ingreso_jefe):,}")

    ingreso_jefe["ingreso_jefe_mensual"] = (
        ingreso_jefe["p524a1"] * ingreso_jefe["p523"].map(FACTOR_MENSUAL)
    )
    n_ingreso_valido = ingreso_jefe["ingreso_jefe_mensual"].notna().sum()
    print(f"De esos, con ingreso mensual valido (P524A1 y P523 no nulos): {n_ingreso_valido:,}")

    # Renombra conglome -> conglomerado para que la llave coincida con el
    # resto del pipeline (que usa COLUMNAS_ID_HOGAR = conglomerado/vivienda/hogar).
    ingreso_jefe = ingreso_jefe.rename(columns={"conglome": "conglomerado"})
    ingreso_jefe["_llave_hogar"] = llave_hogar(ingreso_jefe)
    ingreso_jefe = ingreso_jefe[["_llave_hogar", "ingreso_jefe_mensual"]].drop_duplicates("_llave_hogar")

    COLUMNA_INGRESO_PRINCIPAL = "ingreso_jefe_mensual"
    hogares = hogares.merge(ingreso_jefe, on="_llave_hogar", how="left")
    print(f"Hogares con ingreso del jefe/a (empleado/obrero) disponible: "
          f"{hogares[COLUMNA_INGRESO_PRINCIPAL].notna().sum():,} de {len(hogares):,} "
          f"(el resto -- jefe/a independiente, empleador, no trabaja, etc. -- queda como NaN "
          f"y se excluira de estimar_laids_ingreso.py via dropna).")

    # ============================================================
    # 4. Grilla completa hogar x grupo
    # ============================================================
    print("\n=== Armando grilla completa hogar x grupo ===")
    grilla = hogares.merge(pd.DataFrame({"grupo": GRUPOS}), how="cross")
    grilla = grilla.merge(
        compras[["_llave_hogar", "grupo", "gasto_i", "cantidad_i", "precio_i", "ln_precio_i"]],
        on=["_llave_hogar", "grupo"], how="left"
    )
    grilla["gasto_i"] = grilla["gasto_i"].fillna(0.0)
    grilla["w_i"] = grilla["gasto_i"] / grilla["X"]
    print(f"Filas en la grilla: {len(grilla):,}  ({len(hogares):,} hogares x {len(GRUPOS)} grupos)")

    # ============================================================
    # 5. AJUSTE DE CALIDAD DE DEATON (1988)
    #    Solo sobre precios REALES (hogares que compraron), por grupo.
    #
    #    CORRECCION: Z contiene solo caracteristicas demograficas del
    #    hogar (tamano_hogar, jefe_edad, jefe_sexo). NO se incluye ln_X
    #    (ni ingreso_total_hogar) porque ln_X es la base del indice de
    #    Stone / ln(X/P*), el regresor de interes del sistema AIDS.
    #    Purgar con el propio gasto del hogar introduce una correlacion
    #    mecanica entre precio ajustado y gasto que no tiene contenido
    #    economico (ver Deaton, 1988, "Quality, Quantity, and Spatial
    #    Variation of Price").
    # ============================================================
    print("\n=== Aplicando ajuste de calidad de Deaton (1988) ===")

    VARIABLES_Z = ["tamano_hogar", "jefe_edad", "jefe_sexo"]

    grilla["ln_precio_ajustado"] = grilla["ln_precio_i"]  # por defecto, igual al real

    resumen_ajuste = []

    for grupo in GRUPOS:
        mask = (grilla["grupo"] == grupo) & grilla["ln_precio_i"].notna()
        sub = grilla.loc[mask, ["ln_precio_i"] + VARIABLES_Z].dropna()

        if len(sub) < 30:  # muy pocas compras reales para estimar con confianza
            resumen_ajuste.append({"grupo": grupo, "n": len(sub), "ajustado": "NO (muy pocas obs.)"})
            continue

        Z = sm.add_constant(sub[VARIABLES_Z])
        y = sub["ln_precio_i"]
        modelo = sm.OLS(y, Z).fit()

        Z_bar = sub[VARIABLES_Z].mean()

        idx = sub.index
        desviacion = grilla.loc[idx, VARIABLES_Z] - Z_bar
        ajuste = desviacion.mul(modelo.params[VARIABLES_Z], axis=1).sum(axis=1)

        grilla.loc[idx, "ln_precio_ajustado"] = grilla.loc[idx, "ln_precio_i"] - ajuste

        resumen_ajuste.append({
            "grupo": grupo, "n": len(sub), "ajustado": "SI",
            "r2_regresion_calidad": round(modelo.rsquared, 4)
        })

    tabla_ajuste = pd.DataFrame(resumen_ajuste)
    print(tabla_ajuste.to_string(index=False))
    print("\n(r2_regresion_calidad alto = las caracteristicas del hogar explican bastante")
    print(" del precio pagado -- confirma sesgo de calidad; el ajuste lo esta corrigiendo)")

    # ============================================================
    # 6. Precio de mercado por celda (dominio, mes, grupo) -- YA AJUSTADO
    # ============================================================
    print("\n=== Recalculando precio de mercado e instrumento (con precios ajustados) ===")

    celda = ["dominio", "mes", "grupo"]
    agg_celda = grilla.dropna(subset=["ln_precio_ajustado"]).groupby(celda)["ln_precio_ajustado"].agg(["sum", "count"])
    agg_celda = agg_celda.rename(columns={"sum": "_suma_celda", "count": "_n_celda"})
    grilla = grilla.merge(agg_celda, on=celda, how="left")

    tiene_precio_propio = grilla["ln_precio_ajustado"].notna()
    z_con_compra = (grilla["_suma_celda"] - grilla["ln_precio_ajustado"]) / (grilla["_n_celda"] - 1).clip(lower=1)
    z_sin_compra = grilla["_suma_celda"] / grilla["_n_celda"]

    grilla["z_instrumento"] = np.where(tiene_precio_propio, z_con_compra, z_sin_compra)
    grilla["ln_precio_usado"] = grilla["ln_precio_ajustado"].where(tiene_precio_propio, grilla["z_instrumento"])

    n_imputados = (~tiene_precio_propio).sum()
    print(f"Precios imputados con promedio de mercado: {n_imputados:,} de {len(grilla):,} "
          f"({100*n_imputados/len(grilla):.1f}%)")

    # ============================================================
    # 7. Indice de Stone y ln(X/P*)
    #
    #    NOTA (indice de Stone: pesos propios del hogar vs. pesos fijos):
    #    se probo reemplazar el indice de Stone household-specific
    #    (ln P* = sum_g w_{g,h} * ln p_{g,h}, con el w_i PROPIO de cada
    #    hogar) por un indice tipo Laspeyres con pesos fijos --
    #    participacion de gasto PROMEDIO de la muestra, igual para todos
    #    los hogares-- siguiendo la sugerencia de Moschini (1995, "Units
    #    of Measurement and the Stone Index in Demand System Estimation",
    #    American Journal of Agricultural Economics 77(1): 63-68), que
    #    ademas evita que P* dependa mecanicamente del propio w_i del
    #    hogar (que es la variable dependiente del sistema).
    #
    #    Resultado empirico: el R2 global del sistema empeoro (Berndt's
    #    R2 de -15.6 a -17.5) y el gamma_ii "raro" de pan_basico se hizo
    #    MAS grande (+0.099 -> +0.112), no menor. Con pesos fijos
    #    nacionales, P* deja de reflejar la canasta real de cada hogar en
    #    una poblacion muy heterogenea (ENAHO: costa/sierra/selva, urbano/
    #    rural, deciles de gasto muy distintos), lo que introduce error de
    #    medicion en ln(X/P*) -- un regresor que aparece en las 16
    #    ecuaciones a la vez -- y ese error domina cualquier ganancia de
    #    quitar la endogeneidad mecanica. Por eso se REVIRTIO a los pesos
    #    household-specific (practica estandar en la literatura aplicada
    #    de LA-AIDS, incluyendo el propio Deaton & Muellbauer 1980).
    #
    #    El verdadero problema de los "signos raros" no estaba aqui, sino
    #    en el diagnostico usado para clasificarlos: ver la correccion en
    #    estimar_laids_final.py (elasticidad Marshalliana/Hicksiana
    #    correcta en vez de solo el signo de gamma_ii).
    # ============================================================
    print("\nCalculando indice de Stone (household-specific) y ln(X/P*)...")
    grilla["_w_lnp"] = grilla["w_i"] * grilla["ln_precio_usado"]
    stone = grilla.groupby("_llave_hogar")["_w_lnp"].sum(min_count=1).rename("ln_P_stone")

    columnas_hogar_final = ["X", "tamano_hogar", "jefe_edad", "jefe_sexo", COLUMNA_INGRESO_PRINCIPAL]

    tabla_hogar = hogares.set_index("_llave_hogar")[columnas_hogar_final].join(stone)
    tabla_hogar["ln_X_sobre_P"] = np.log(tabla_hogar["X"]) - tabla_hogar["ln_P_stone"]

    # ============================================================
    # 8. Formato ANCHO
    # ============================================================
    print("\nConvirtiendo a formato ancho...")
    w_ancho = grilla.pivot_table(index="_llave_hogar", columns="grupo", values="w_i")
    w_ancho.columns = [f"w_{c}" for c in w_ancho.columns]

    p_ancho = grilla.pivot_table(index="_llave_hogar", columns="grupo", values="ln_precio_usado")
    p_ancho.columns = [f"ln_precio_{c}" for c in p_ancho.columns]

    z_ancho = grilla.pivot_table(index="_llave_hogar", columns="grupo", values="z_instrumento")
    z_ancho.columns = [f"z_{c}" for c in z_ancho.columns]

    base_final = pd.concat([w_ancho, p_ancho, z_ancho, tabla_hogar], axis=1).reset_index()
    base_final[COLUMNAS_ID_HOGAR] = base_final["_llave_hogar"].str.split("_", expand=True)
    base_final = base_final.drop(columns="_llave_hogar")

    base_final.to_parquet(SALIDA, index=False)
    print(f"\nBase final guardada en: {SALIDA}  ({len(base_final):,} hogares)")

    n_completos = base_final.dropna(subset=[c for c in base_final.columns if c.startswith(("w_", "ln_precio_", "z_"))]).shape[0]
    print(f"Hogares con datos completos en los {len(GRUPOS)} grupos: {n_completos:,} "
          f"({100*n_completos/len(base_final):.1f}%)")


if FORZAR_RECONSTRUCCION or not all(os.path.exists(_a) for _a in _SALIDAS_paso10_base_final):
    _ejecutar_paso10_base_final()
else:
    print(f"[SALTADO] Ya existen: {_SALIDAS_paso10_base_final} -- no se reconstruye este paso.")
    print("(Pon FORZAR_RECONSTRUCCION = True en la celda de configuracion si quieres forzar el rebuild.)")


[SALTADO] Ya existen: ['base_laids_final.parquet'] -- no se reconstruye este paso.
(Pon FORZAR_RECONSTRUCCION = True en la celda de configuracion si quieres forzar el rebuild.)


## Parte 2: Regresiones y simulacion de politica

Estas celdas **siempre se ejecutan** (no tienen checkpoint): son
rapidas porque solo leen `base_laids_final.parquet` (ya construido en
la Parte 1) y corren la regresion IV-SUR, que toma segundos.

1. **Sistema LA-AIDS base** (`estimar_laids_final.py`): elasticidades
   precio-propio (Marshalliana/Hicksiana) y elasticidad-gasto por
   grupo, sin efecto de ingreso.
2. **Sistema LA-AIDS con efecto de ingreso** (`estimar_laids_ingreso.py`):
   agrega `theta_i * ln(ingreso del jefe/a)` a cada ecuacion, muestra
   completa y submuestra de bajos ingresos (<= RMV).
3. **Simulacion del aumento de RMV** (`simular_impacto_rmv.py`): usa el
   theta_i de la submuestra de bajos ingresos para simular el efecto de
   subir la RMV de S/ 1,130 a S/ 1,300 sobre la composicion de la
   canasta de los hogares afectados.

### 2.1 -- Sistema LA-AIDS base (sin efecto de ingreso)

In [14]:
"""
estimar_laids_final.py
--------------------------
Estimacion FINAL del sistema LA-AIDS:
  - 17 grupos (con pan y pollo subdivididos)
  - precios ajustados por calidad (Deaton 1988)
  - solo precio propio endogeno por ecuacion, instrumentado con su
    z_i leave-one-out (diseño exactamente identificado)
  - parametros implicitos del grupo de referencia via adding-up

DIAGNOSTICO CORREGIDO (ver seccion final):
  El chequeo original de "signo raro" verificaba solo el signo del
  coeficiente crudo gamma_ii (RARO si gamma_ii > 0). Esa NO es la
  prueba teorica correcta en AIDS. La elasticidad-precio propia
  (la que realmente tiene que ser negativa por teoria de la demanda)
  es, siguiendo Deaton & Muellbauer (1980, "An Almost Ideal Demand
  System", American Economic Review 70(3): 312-326):

    Marshalliana (no compensada):  e_ii = -1 + gamma_ii/w_i - beta_i
    Hicksiana (compensada):        e_ii* = gamma_ii/w_i - (1 - w_i)

  donde w_i es la participacion de gasto PROMEDIO del grupo (no el
  coeficiente). Para bienes con participacion de gasto grande (p.ej.
  pan_basico, ~20% del gasto en alimentos), el termino "-1" y "-beta_i"
  ya empujan la elasticidad hacia terreno muy negativo antes de sumar
  gamma_ii/w_i, asi que un gamma_ii positivo NO implica necesariamente
  una demanda con pendiente positiva. El chequeo de signo crudo solo
  es un buen atajo cuando w_i es chico (ahi gamma_ii/w_i domina).
  Por eso el diagnostico se reemplaza por el signo de e_ii* (Hicksiana),
  que es la que exige la teoria (matriz de Slutsky semidefinida
  negativa), y se reporta tambien la Marshalliana como referencia.
"""

import pandas as pd
import numpy as np
from linearmodels.system import IV3SLS
import pyarrow.parquet as pq  # <-- agregar este import

ARCHIVO = "base_laids_final.parquet"
GRUPO_REFERENCIA = "verduras_hortalizas"

df = pq.read_table(ARCHIVO).to_pandas()  # <-- en vez de pd.read_parquet(ARCHIVO)

grupos = sorted(c.replace("w_", "") for c in df.columns if c.startswith("w_"))
print(f"Grupos detectados ({len(grupos)}): {grupos}")

if GRUPO_REFERENCIA not in grupos:
    GRUPO_REFERENCIA = grupos[0]
    print(f"Aviso: grupo de referencia no encontrado, usando '{GRUPO_REFERENCIA}' en su lugar.")

grupos_ecuaciones = [g for g in grupos if g != GRUPO_REFERENCIA]

columnas_precio = [f"ln_precio_{g}" for g in grupos]
columnas_instrumento = [f"z_{g}" for g in grupos]
columnas_w = [f"w_{g}" for g in grupos]

columnas_necesarias = columnas_precio + columnas_instrumento + columnas_w + ["ln_X_sobre_P"]
df_completo = df.dropna(subset=columnas_necesarias).copy()
print(f"\nHogares con datos completos: {len(df_completo):,} de {len(df):,}")

# Participacion de gasto promedio por grupo, EN LA MUESTRA DE ESTIMACION
# (df_completo) -- se usa solo para el diagnostico de elasticidades,
# no entra en la estimacion del sistema.
w_bar = df_completo[columnas_w].mean()
w_bar.index = [c.replace("w_", "") for c in w_bar.index]

# ============================================================
# Sistema: solo precio propio endogeno por ecuacion
# ============================================================
ecuaciones = {}
for g in grupos_ecuaciones:
    dependiente = df_completo[f"w_{g}"]
    precios_ajenos = [f"ln_precio_{k}" for k in grupos if k != g]

    exog = df_completo[["ln_X_sobre_P"] + precios_ajenos].copy()
    exog["const"] = 1.0

    endog = df_completo[[f"ln_precio_{g}"]].copy()
    instrumentos = df_completo[[f"z_{g}"]].copy()

    ecuaciones[g] = {"dependent": dependiente, "exog": exog, "endog": endog, "instruments": instrumentos}

print(f"\nEstimando sistema IV-SUR (3SLS), {len(grupos_ecuaciones)} ecuaciones, "
      f"referencia: {GRUPO_REFERENCIA}...")

modelo = IV3SLS(ecuaciones)
resultado = modelo.fit(cov_type="robust")

print("\n" + "=" * 70)
print(resultado)

with open("resultados_laids_final.txt", "w", encoding="utf-8") as f:
    f.write(str(resultado))
print("\nResultados completos guardados en: resultados_laids_final.txt")

# ============================================================
# Resumen: beta (elasticidad-gasto) y elasticidad-precio propia
# CORREGIDA (Marshalliana y Hicksiana), no solo el signo de gamma_ii
# ============================================================
print("\n=== RESUMEN: beta_i (gasto) y elasticidad-precio propia por grupo ===")
print(f"{'Grupo':22s} {'w_bar':>7s} {'beta':>10s} {'gamma_ii':>10s} "
      f"{'e_Marshall':>11s} {'e_Hicks':>9s}  Diagnostico")
print("-" * 95)
for g in grupos_ecuaciones:
    beta = resultado.params.get(f"{g}_ln_X_sobre_P", np.nan)
    gamma = resultado.params.get(f"{g}_ln_precio_{g}", np.nan)
    wi = w_bar.get(g, np.nan)

    e_marshall = -1 + gamma / wi - beta
    e_hicks = gamma / wi - (1 - wi)

    diagnostico = "OK (Hicks<0)" if e_hicks < 0 else "REVISAR (Hicks>0)"

    print(f"{g:22s} {wi:7.4f} {beta:+10.4f} {gamma:+10.4f} "
          f"{e_marshall:+11.4f} {e_hicks:+9.4f}  {diagnostico}")

print("\n(e_Marshall = -1 + gamma_ii/w_i - beta_i;  e_Hicks = gamma_ii/w_i - (1-w_i);")
print(" formulas de Deaton & Muellbauer 1980. La que exige la teoria -- Slutsky")
print(" semidefinida negativa -- es e_Hicks < 0. El signo crudo de gamma_ii NO es")
print(" el test correcto, especialmente para grupos con w_i grande como pan_basico.)")

# ============================================================
# Parametros implicitos del grupo de referencia (adding-up)
# ============================================================
print(f"\n=== Parametros implicitos de '{GRUPO_REFERENCIA}' ===")
params = resultado.params
cov = resultado.cov

def implicito_con_se(nombres):
    nombres = [n for n in nombres if n in params.index]
    if not nombres:
        return None, None
    valor = -params[nombres].sum()
    submatriz = cov.loc[nombres, nombres]
    varianza = submatriz.values.sum()
    se = np.sqrt(varianza) if varianza > 0 else np.nan
    return valor, se

alpha_ref, se_alpha_ref = implicito_con_se([f"{g}_const" for g in grupos_ecuaciones])
if alpha_ref is not None:
    alpha_ref += 1
beta_ref, se_beta_ref = implicito_con_se([f"{g}_ln_X_sobre_P" for g in grupos_ecuaciones])

nombres_gamma_ref = []
for g in grupos_ecuaciones:
    nombre = f"{g}_ln_precio_{GRUPO_REFERENCIA}"
    if nombre in params.index:
        nombres_gamma_ref.append(nombre)
gamma_ref_ref, se_gamma_ref_ref = implicito_con_se(nombres_gamma_ref)

print(f"alpha_{GRUPO_REFERENCIA} = {alpha_ref:.4f}  (SE={se_alpha_ref:.4f})")
if beta_ref is not None:
    tipo_ref = "LUJO/NORMAL SUPERIOR" if beta_ref > 0 else "NECESARIO/INFERIOR"
    z_beta = beta_ref / se_beta_ref if se_beta_ref else np.nan
    print(f"beta_{GRUPO_REFERENCIA}  = {beta_ref:+.4f}  (SE={se_beta_ref:.4f}, z={z_beta:.2f})  ({tipo_ref})")
if gamma_ref_ref is not None:
    wi_ref = w_bar.get(GRUPO_REFERENCIA, np.nan)
    e_marshall_ref = -1 + gamma_ref_ref / wi_ref - (beta_ref if beta_ref is not None else np.nan)
    e_hicks_ref = gamma_ref_ref / wi_ref - (1 - wi_ref)
    print(f"gamma_{GRUPO_REFERENCIA},{GRUPO_REFERENCIA} (precio propio implicito) = "
          f"{gamma_ref_ref:+.4f}  (SE={se_gamma_ref_ref:.4f})")
    print(f"e_Marshall_{GRUPO_REFERENCIA} = {e_marshall_ref:+.4f}   "
          f"e_Hicks_{GRUPO_REFERENCIA} = {e_hicks_ref:+.4f}")

# ============================================================
# Chequeo de consistencia
# ============================================================
suma_w = df_completo[columnas_w].sum(axis=1)
print(f"\n=== Chequeo: suma de w_i (debe ser ~1.0) ===")
print(f"Media: {suma_w.mean():.6f}  |  Min: {suma_w.min():.6f}  |  Max: {suma_w.max():.6f}")

Grupos detectados (17): ['aceites_grasas', 'azucar_dulces', 'bebidas_alcoholicas', 'bebidas_no_alcoholicas', 'cafe_te_infusiones', 'carnes_rojas', 'comida_fuera_hogar', 'frutas', 'frutos_secos_semillas', 'lacteos_huevos', 'menestras', 'menudencia_aves', 'pan_basico', 'pasteleria_dulce', 'pescados_mariscos', 'pollo_piezas', 'verduras_hortalizas']

Hogares con datos completos: 27,164 de 33,418

Estimando sistema IV-SUR (3SLS), 16 ecuaciones, referencia: verduras_hortalizas...

                           System GLS Estimation Summary                           
Estimator:                        GLS   Overall R-squared:                  -0.1409
No. Equations.:                    16   McElroy's R-squared:                -0.0573
No. Observations:               27164   Judge's (OLS) R-squared:            -0.1409
Date:                Sun, Aug 16 2026   Berndt's R-squared:                 -15.572
Time:                        15:26:37   Dhrymes's R-squared:                -0.1409
                

### 2.2 -- Sistema LA-AIDS con efecto del ingreso del jefe/a de hogar

In [16]:
"""
estimar_laids_ingreso.py
--------------------------
Extension del sistema LA-AIDS final (estimar_laids_final.py) para medir
el efecto del INGRESO sobre la composicion de la canasta de alimentos,
mas alla de lo que ya explica el gasto total en alimentos ln(X/P).

Se agrega theta_i * ln(ingreso) a las 16 ecuaciones -- un shifter tipo
"AIDS con traduccion demografica" (Pollak & Wales, 1981; Ray, 1983):

    w_i = alpha_i + sum_j gamma_ij*ln(p_j) + beta_i*ln(X/P)
          + theta_i*ln(ingreso) + error

Si theta_i es significativo para algun grupo, eso rechaza el supuesto
de separabilidad en dos etapas para ese grupo: el ingreso mueve la
composicion de la canasta incluso controlando por cuanto ya se gasta
en alimentos en total.

Se corre DOS VECES:
  1. Muestra completa (referencia).
  2. Muestra restringida a hogares con ingreso_jefe_mensual
     <= RMV_PERU (S/ 1,130) -- el segmento de interes principal segun
     lo pedido.

IMPORTANTE -- variable de ingreso:
  "ingreso_jefe_mensual" es el ingreso INDIVIDUAL del jefe/a de hogar
  en su ocupacion principal, restringido a jefes/as que son
  empleado/obrero (P507 = 3 o 4 en el modulo 500 de la ENAHO),
  construido a partir de P524A1 (ingreso en la frecuencia declarada)
  convertido a monto mensual segun P523. Jefes/as con otra categoria
  ocupacional (independiente, empleador, trabajador familiar no
  remunerado, no trabaja, etc.) quedan como NaN aqui -- por diseno,
  ya que la RMV es un umbral legal para trabajadores dependientes.
  Ver construir_base_laids_final.py, seccion 3B, para el detalle del
  cruce (modulos 200 y 500 crudos de la ENAHO).
"""
import pandas as pd
import numpy as np
from linearmodels.system import IV3SLS
import pyarrow.parquet as pq  # <-- agregar

ARCHIVO = "base_laids_final.parquet"
GRUPO_REFERENCIA = "verduras_hortalizas"

COLUMNA_INGRESO_PRINCIPAL = "ingreso_jefe_mensual"

RMV_PERU = 1130

df = pq.read_table(ARCHIVO).to_pandas()  # <-- en vez de pd.read_parquet(ARCHIVO)

grupos = sorted(c.replace("w_", "") for c in df.columns if c.startswith("w_"))
grupos_ecuaciones = [g for g in grupos if g != GRUPO_REFERENCIA]
columnas_precio = [f"ln_precio_{g}" for g in grupos]
columnas_instrumento = [f"z_{g}" for g in grupos]
columnas_w = [f"w_{g}" for g in grupos]

if COLUMNA_INGRESO_PRINCIPAL not in df.columns:
    candidatas = [c for c in df.columns if "ingreso" in c.lower()]
    raise ValueError(
        f"No se encontro la columna '{COLUMNA_INGRESO_PRINCIPAL}' en {ARCHIVO}.\n"
        f"Columnas disponibles que contienen 'ingreso': {candidatas}\n"
        f"Verifica que construir_base_laids_final.py haya corrido con el nombre correcto "
        f"de columna (variable COLUMNA_INGRESO_PRINCIPAL en ese script) y que ambos "
        f"scripts usen el MISMO nombre."
    )

col_ingreso = COLUMNA_INGRESO_PRINCIPAL
print(f"Usando columna de ingreso: '{col_ingreso}' (ingreso individual del jefe/a de "
      f"hogar, solo empleado/obrero, convertido a mensual).")

df["ln_ingreso"] = np.where(df[col_ingreso] > 0, np.log(df[col_ingreso]), np.nan)

columnas_necesarias = columnas_precio + columnas_instrumento + columnas_w + ["ln_X_sobre_P", "ln_ingreso"]
df_completo = df.dropna(subset=columnas_necesarias).copy()
print(f"Hogares con datos completos (incluyendo ingreso): {len(df_completo):,} de {len(df):,}")

n_bajo_rmv = (df_completo[col_ingreso] <= RMV_PERU).sum()
print(f"De esos, con {col_ingreso} <= S/ {RMV_PERU}: {n_bajo_rmv:,} "
      f"({100*n_bajo_rmv/len(df_completo):.1f}%)")


def construir_ecuaciones(data, grupos_ecuaciones, grupos):
    ecuaciones = {}
    for g in grupos_ecuaciones:
        dependiente = data[f"w_{g}"]
        precios_ajenos = [f"ln_precio_{k}" for k in grupos if k != g]

        exog = data[["ln_X_sobre_P", "ln_ingreso"] + precios_ajenos].copy()
        exog["const"] = 1.0

        endog = data[[f"ln_precio_{g}"]].copy()
        instrumentos = data[[f"z_{g}"]].copy()

        ecuaciones[g] = {"dependent": dependiente, "exog": exog, "endog": endog, "instruments": instrumentos}
    return ecuaciones


def estimar_y_resumir(data, etiqueta):
    print(f"\n{'=' * 70}\nEstimando sistema IV-SUR (3SLS) -- muestra: {etiqueta} "
          f"({len(data):,} hogares)\n{'=' * 70}")

    ecuaciones = construir_ecuaciones(data, grupos_ecuaciones, grupos)
    modelo = IV3SLS(ecuaciones)
    resultado = modelo.fit(cov_type="robust")

    nombre_archivo = f"resultados_laids_ingreso_{etiqueta}.txt"
    with open(nombre_archivo, "w", encoding="utf-8") as f:
        f.write(str(resultado))
    print(f"Resultados completos guardados en: {nombre_archivo}")

    w_bar = data[columnas_w].mean()
    w_bar.index = [c.replace("w_", "") for c in w_bar.index]

    print(f"\n--- theta_i (efecto del ingreso, controlando por X) -- muestra {etiqueta} ---")
    print(f"{'Grupo':22s} {'w_bar':>7s} {'beta':>10s} {'theta_i':>10s} {'p-valor':>9s}  Interpretacion")
    print("-" * 90)

    theta_dict = {}
    for g in grupos_ecuaciones:
        beta = resultado.params.get(f"{g}_ln_X_sobre_P", np.nan)
        theta = resultado.params.get(f"{g}_ln_ingreso", np.nan)
        pval = resultado.pvalues.get(f"{g}_ln_ingreso", np.nan)
        wi = w_bar.get(g, np.nan)
        theta_dict[g] = theta

        if pval < 0.05:
            interp = "significativo (rechaza separabilidad)" if abs(theta) > 0 else "significativo"
        else:
            interp = "no significativo (consistente con separabilidad)"

        print(f"{g:22s} {wi:7.4f} {beta:+10.4f} {theta:+10.4f} {pval:9.4f}  {interp}")

    # Theta implicito del grupo de referencia via adding-up: sum_i theta_i = 0
    theta_ref = -sum(theta_dict.values())
    print(f"\ntheta_{GRUPO_REFERENCIA} (implicito, adding-up) = {theta_ref:+.4f}")

    n_sig = sum(1 for g in grupos_ecuaciones
                if resultado.pvalues.get(f"{g}_ln_ingreso", 1.0) < 0.05)
    print(f"\nGrupos con theta_i significativo al 5% (de {len(grupos_ecuaciones)}): {n_sig}")

    return resultado, theta_dict


# ============================================================
# 1. Muestra completa (referencia)
# ============================================================
resultado_completo, theta_completo = estimar_y_resumir(df_completo, "muestra_completa")

# ============================================================
# 2. Muestra restringida: ingreso principal <= RMV
# ============================================================
df_bajo_rmv = df_completo[df_completo[col_ingreso] <= RMV_PERU].copy()

if len(df_bajo_rmv) < 500:
    print(f"\nAviso: solo {len(df_bajo_rmv)} hogares con {col_ingreso} <= {RMV_PERU}. "
          f"La estimacion en este subgrupo puede ser poco precisa (errores estandar grandes).")

resultado_rmv, theta_rmv = estimar_y_resumir(df_bajo_rmv, "ingreso_bajo_RMV")

# ============================================================
# 3. Comparacion lado a lado
# ============================================================
print(f"\n{'=' * 70}\nCOMPARACION: theta_i, muestra completa vs. ingreso <= RMV (S/ {RMV_PERU})"
      f"\n{'=' * 70}")
print(f"{'Grupo':22s} {'theta (completa)':>18s} {'theta (<=RMV)':>16s} {'Diferencia':>12s}")
print("-" * 72)
for g in grupos_ecuaciones:
    t_full = theta_completo.get(g, np.nan)
    t_rmv = theta_rmv.get(g, np.nan)
    print(f"{g:22s} {t_full:+18.4f} {t_rmv:+16.4f} {t_rmv - t_full:+12.4f}")

print("\n(theta_i > 0 y significativo: a igual gasto total en alimentos, mas ingreso")
print(" se asocia a MAYOR participacion de ese grupo en la canasta. theta_i < 0: menor")
print(" participacion. Comparar columnas para ver si el efecto del ingreso es distinto")
print(" -- tipicamente mas fuerte -- en el segmento de bajos ingresos.)")

Usando columna de ingreso: 'ingreso_jefe_mensual' (ingreso individual del jefe/a de hogar, solo empleado/obrero, convertido a mensual).
Hogares con datos completos (incluyendo ingreso): 7,508 de 33,418
De esos, con ingreso_jefe_mensual <= S/ 1130: 2,064 (27.5%)

Estimando sistema IV-SUR (3SLS) -- muestra: muestra_completa (7,508 hogares)
Resultados completos guardados en: resultados_laids_ingreso_muestra_completa.txt

--- theta_i (efecto del ingreso, controlando por X) -- muestra muestra_completa ---
Grupo                    w_bar       beta    theta_i   p-valor  Interpretacion
------------------------------------------------------------------------------------------
aceites_grasas          0.0386    -0.0031    -0.0039    0.0056  significativo (rechaza separabilidad)
azucar_dulces           0.0557    -0.0007    -0.0030    0.0055  significativo (rechaza separabilidad)
bebidas_alcoholicas     0.0005    -0.0004    -0.0001    0.4944  no significativo (consistente con separabilidad)
bebidas

### 2.3 -- Simulacion: aumento de la RMV de S/ 1,130 a S/ 1,300

In [17]:
"""
simular_impacto_rmv.py
--------------------------
Simulacion del efecto de subir la Remuneracion Minima Vital (RMV) de
S/ 1,130 a S/ 1,300 sobre la composicion de la canasta de alimentos de
los hogares potencialmente afectados.

ALCANCE (decidido explicitamente por el usuario) -- IMPORTANTE:
  Esta simulacion usa UNICAMENTE el efecto-ingreso theta_i estimado en
  estimar_laids_ingreso.py (el shifter theta_i*ln(ingreso) agregado a
  las ecuaciones de participacion del gasto, controlando por ln(X/P)).

  Es decir: el gasto TOTAL en alimentos (X) se mantiene FIJO. Lo que
  esta simulacion muestra es como se REACOMODARIA la canasta -- que
  grupos ganan y cuales pierden participacion -- si el ingreso del
  jefe/a de hogar subiera, a gasto total en alimentos constante.

  Esto NO captura el canal adicional por el cual un mayor ingreso
  tipicamente tambien eleva el gasto total en alimentos (X sube, no
  solo su composicion) -- ese seria un "Engel curve" / primera etapa
  del presupuesto, que el usuario decidio dejar fuera por ahora:

    "ok mira, mejor solo nos quedamos con el impacto del ingreso a
     secas y ya yo evaluo el efecto de pasar de un sueldo de 1130
     hasta 1299 fijo a 1300."

  Por diseno del sistema AIDS (adding-up: suma_i theta_i = 0), la suma
  de los cambios en participacion (delta_w_i) a traves de los 17 grupos
  es ~0 por construccion: es una REDISTRIBUCION del presupuesto de
  alimentos, no un crecimiento del presupuesto. Esto se verifica
  explicitamente en la salida.

POBLACION AFECTADA (definicion operativa):
  Hogares cuyo jefe/a de hogar es empleado/obrero (P507 = 3 o 4) y
  gana actualmente entre S/ 1,130 y S/ 1,299 (banda [RMV_VIEJA,
  RMV_NUEVA - 1]). Se asume que la politica los sube a exactamente
  S/ 1,300 (RMV_NUEVA). Hogares con ingreso ya >= 1,300 no se tocan;
  hogares con ingreso < 1,130 tampoco (no son el margen directamente
  afectado por ESTE incremento especifico, aunque en la practica podrian
  estar sub-declarando o en informalidad parcial -- fuera del alcance
  aqui).

THETA_I USADO:
  Se re-estima el sistema con el shifter de ingreso restringido a la
  submuestra ya validada "ingreso <= RMV_PERU (1130)" -- el segmento de
  bajos ingresos mas cercano a la poblacion afectada por el aumento de
  RMV -- en lugar de la muestra completa. La logica: el efecto del
  ingreso sobre la composicion de la canasta puede no ser lineal /
  constante en todo el rango de ingresos (hogares ricos vs. hogares
  cerca del salario minimo pueden reaccionar distinto), y el segmento
  de interes para esta politica es precisamente el de bajos ingresos.
  Se reporta tambien, como referencia, que hubiese dado usar el theta_i
  de la muestra completa.

  Se muestran DOS versiones del resultado:
    (a) "Punto central": usa el theta_i estimado tal cual, aunque no
        sea significativo al 5% para algunos grupos.
    (b) "Conservador": pone en 0 los theta_i NO significativos al 5%,
        para no sobre-interpretar ruido de estimacion en grupos donde
        no hay evidencia solida de que el ingreso mueva la canasta.
"""

import pandas as pd
import numpy as np
from linearmodels.system import IV3SLS
import pyarrow.parquet as pq  # <-- agregar

ARCHIVO = "base_laids_final.parquet"
GRUPO_REFERENCIA = "verduras_hortalizas"

COLUMNA_INGRESO_PRINCIPAL = "ingreso_jefe_mensual"

RMV_VIEJA = 1130.0
RMV_NUEVA = 1300.0

ALFA_SIGNIFICANCIA = 0.05

# ------------------------------------------------------------------
# 0. Cargar datos y preparar
# ------------------------------------------------------------------
df = pq.read_table(ARCHIVO).to_pandas()  # <-- en vez de pd.read_parquet(ARCHIVO)

grupos = sorted(c.replace("w_", "") for c in df.columns if c.startswith("w_"))
grupos_ecuaciones = [g for g in grupos if g != GRUPO_REFERENCIA]
columnas_precio = [f"ln_precio_{g}" for g in grupos]
columnas_instrumento = [f"z_{g}" for g in grupos]
columnas_w = [f"w_{g}" for g in grupos]

if COLUMNA_INGRESO_PRINCIPAL not in df.columns:
    raise ValueError(
        f"No se encontro la columna '{COLUMNA_INGRESO_PRINCIPAL}' en {ARCHIVO}. "
        f"Verifica que construir_base_laids_final.py (seccion 3B) haya corrido."
    )

col_ingreso = COLUMNA_INGRESO_PRINCIPAL
df["ln_ingreso"] = np.where(df[col_ingreso] > 0, np.log(df[col_ingreso]), np.nan)

columnas_necesarias = columnas_precio + columnas_instrumento + columnas_w + ["ln_X_sobre_P", "ln_ingreso", "X"]
df_completo = df.dropna(subset=columnas_necesarias).copy()
print(f"Hogares con datos completos (incluyendo ingreso del jefe/a, empleado/obrero): "
      f"{len(df_completo):,} de {len(df):,}")


# ------------------------------------------------------------------
# 1. Re-estimar theta_i en la submuestra de bajos ingresos (<=RMV_VIEJA)
# ------------------------------------------------------------------
def construir_ecuaciones(data, grupos_ecuaciones, grupos):
    ecuaciones = {}
    for g in grupos_ecuaciones:
        dependiente = data[f"w_{g}"]
        precios_ajenos = [f"ln_precio_{k}" for k in grupos if k != g]

        exog = data[["ln_X_sobre_P", "ln_ingreso"] + precios_ajenos].copy()
        exog["const"] = 1.0

        endog = data[[f"ln_precio_{g}"]].copy()
        instrumentos = data[[f"z_{g}"]].copy()

        ecuaciones[g] = {"dependent": dependiente, "exog": exog, "endog": endog, "instruments": instrumentos}
    return ecuaciones


def estimar_theta(data, etiqueta):
    print(f"\n{'=' * 70}\nEstimando theta_i -- muestra: {etiqueta} ({len(data):,} hogares)\n{'=' * 70}")
    ecuaciones = construir_ecuaciones(data, grupos_ecuaciones, grupos)
    modelo = IV3SLS(ecuaciones)
    resultado = modelo.fit(cov_type="robust")

    theta_dict, pval_dict = {}, {}
    for g in grupos_ecuaciones:
        theta_dict[g] = resultado.params.get(f"{g}_ln_ingreso", np.nan)
        pval_dict[g] = resultado.pvalues.get(f"{g}_ln_ingreso", np.nan)

    theta_dict[GRUPO_REFERENCIA] = -sum(theta_dict.values())
    # p-valor del theta implicito del grupo de referencia: no se calcula
    # directamente (requeriria la covarianza conjunta); se deja como NaN
    # y se trata como "no significativo" por defecto en la version
    # conservadora, salvo que se calcule aparte.
    pval_dict[GRUPO_REFERENCIA] = np.nan

    return resultado, theta_dict, pval_dict


df_bajo_rmv = df_completo[df_completo[col_ingreso] <= RMV_VIEJA].copy()
if len(df_bajo_rmv) < 500:
    print(f"\nAviso: solo {len(df_bajo_rmv)} hogares con {col_ingreso} <= {RMV_VIEJA}. "
          f"La estimacion en este subgrupo puede ser poco precisa.")

_, theta_rmv, pval_rmv = estimar_theta(df_bajo_rmv, f"ingreso_bajo_RMV (<= {RMV_VIEJA:.0f})")

# Referencia informativa: theta_i en la muestra completa (no se usa en
# la simulacion, solo se imprime para comparar).
_, theta_completo, pval_completo = estimar_theta(df_completo, "muestra_completa (referencia)")

print(f"\n{'=' * 70}\nComparacion theta_i: submuestra <=RMV (usada en la simulacion) vs. "
      f"muestra completa (referencia)\n{'=' * 70}")
print(f"{'Grupo':22s} {'theta (<=RMV)':>14s} {'p-valor':>9s}  {'theta (completa)':>17s} {'p-valor':>9s}")
print("-" * 78)
for g in grupos:
    tr, pr = theta_rmv.get(g, np.nan), pval_rmv.get(g, np.nan)
    tc, pc = theta_completo.get(g, np.nan), pval_completo.get(g, np.nan)
    marca_r = "*" if (not np.isnan(pr) and pr < ALFA_SIGNIFICANCIA) else " "
    marca_c = "*" if (not np.isnan(pc) and pc < ALFA_SIGNIFICANCIA) else " "
    print(f"{g:22s} {tr:+14.4f}{marca_r} {pr:9.4f}  {tc:+17.4f}{marca_c} {pc:9.4f}")
print("(* = significativo al 5%. theta del grupo de referencia es implicito por adding-up; su p-valor no se reporta.)")


# ------------------------------------------------------------------
# 2. Definir poblacion afectada y simular
# ------------------------------------------------------------------
afectados = df_completo[
    (df_completo[col_ingreso] >= RMV_VIEJA) & (df_completo[col_ingreso] < RMV_NUEVA)
].copy()
print(f"\n{'=' * 70}\nPoblacion afectada: jefe/a empleado/obrero con ingreso en "
      f"[S/ {RMV_VIEJA:.0f}, S/ {RMV_NUEVA:.0f})\n{'=' * 70}")
print(f"Hogares afectados: {len(afectados):,} de {len(df_completo):,} "
      f"({100 * len(afectados) / len(df_completo):.1f}% de la muestra con datos completos)")

if len(afectados) == 0:
    raise SystemExit("No hay hogares en la banda de ingreso especificada -- revisar RMV_VIEJA/RMV_NUEVA o los datos.")

afectados["ingreso_actual"] = afectados[col_ingreso]
afectados["ingreso_nuevo"] = RMV_NUEVA
afectados["delta_ln_ingreso"] = np.log(afectados["ingreso_nuevo"]) - np.log(afectados["ingreso_actual"])

print(f"Ingreso actual promedio (afectados): S/ {afectados['ingreso_actual'].mean():,.0f}")
print(f"Incremento promedio de ingreso: S/ {(afectados['ingreso_nuevo'] - afectados['ingreso_actual']).mean():,.0f} "
      f"({100 * afectados['delta_ln_ingreso'].mean():.1f}% en log-puntos promedio)")


def simular(theta_dict, pval_dict, etiqueta, conservador):
    """
    Aplica delta_w_i = theta_i * delta_ln_ingreso a cada hogar afectado.
    Si conservador=True, theta_i no significativo al 5% se trata como 0.
    X (gasto total en alimentos) se mantiene FIJO -- ver docstring.
    """
    filas = []
    suma_delta_w_pp = 0.0  # para chequeo de adding-up (debe dar ~0)
    for g in grupos:
        theta = theta_dict.get(g, np.nan)
        pval = pval_dict.get(g, np.nan)
        sig = (not np.isnan(pval)) and (pval < ALFA_SIGNIFICANCIA)

        # Nota: el theta del grupo de referencia no tiene p-valor propio
        # (es implicito por adding-up), asi que en la version conservadora
        # se usa tal cual -- ponerlo en 0 arbitrariamente rompería el
        # adding-up (sum theta_i = 0) de la simulacion.
        if conservador and g != GRUPO_REFERENCIA and not sig:
            theta_usado = 0.0
        else:
            theta_usado = theta

        delta_w = theta_usado * afectados["delta_ln_ingreso"]
        w_actual = afectados[f"w_{g}"]
        w_nuevo = w_actual + delta_w
        gasto_actual = w_actual * afectados["X"]
        gasto_nuevo = w_nuevo * afectados["X"]
        delta_gasto = gasto_nuevo - gasto_actual

        suma_delta_w_pp += delta_w.mean()

        filas.append({
            "grupo": g,
            "theta_i": theta,
            "p_valor": pval,
            "significativo_5pct": sig if g != GRUPO_REFERENCIA else "(implicito, adding-up)",
            "theta_usado_en_simulacion": theta_usado,
            "w_actual_promedio": w_actual.mean(),
            "w_nuevo_promedio": w_nuevo.mean(),
            "delta_w_pp_promedio": delta_w.mean() * 100,
            "gasto_actual_promedio_soles_mes": gasto_actual.mean(),
            "gasto_nuevo_promedio_soles_mes": gasto_nuevo.mean(),
            "delta_gasto_promedio_soles_mes": delta_gasto.mean(),
            "delta_gasto_total_muestra_soles_mes": delta_gasto.sum(),
        })

    resumen = pd.DataFrame(filas).sort_values("delta_w_pp_promedio", ascending=False).reset_index(drop=True)

    print(f"\n{'=' * 100}\nSIMULACION -- version: {etiqueta}\n{'=' * 100}")
    print(f"{'Grupo':22s} {'theta usado':>12s} {'sig.5%':>7s} {'w_actual':>9s} {'w_nuevo':>9s} "
          f"{'delta_w(pp)':>12s} {'delta_gasto/hog (S/)':>21s}")
    print("-" * 100)
    for _, r in resumen.iterrows():
        sig_str = str(r["significativo_5pct"])[:7] if r["grupo"] != GRUPO_REFERENCIA else "implic."
        print(f"{r['grupo']:22s} {r['theta_usado_en_simulacion']:+12.4f} {sig_str:>7s} "
              f"{r['w_actual_promedio']:9.4f} {r['w_nuevo_promedio']:9.4f} "
              f"{r['delta_w_pp_promedio']:+12.3f} {r['delta_gasto_promedio_soles_mes']:+21.2f}")

    print(f"\nChequeo adding-up: suma de delta_w_i promedio a traves de los {len(grupos)} grupos "
          f"= {suma_delta_w_pp * 100:+.6f} pp (debe ser ~0 -- confirma que X no cambia, solo la "
          f"composicion de la canasta se redistribuye).")
    if conservador and abs(suma_delta_w_pp) > 1e-9:
        print("Nota: en la version 'conservador' esta suma NO da exactamente 0. Motivo: el theta "
              "implicito del grupo de referencia se calculo por adding-up usando los theta_i ORIGINALES "
              "(sin poner en 0 los no significativos); al poner en 0 varios theta_i de las otras 16 "
              "ecuaciones pero mantener el de referencia sin tocar, se rompe la identidad sum(theta_i)=0. "
              "Es un efecto de puesta a 0 mecanica de los theta_i con menor evidencia, no un error de "
              "calculo -- se reporta tal cual para que quede transparente.")

    nombre_csv = f"simulacion_rmv_{etiqueta.replace(' ', '_').lower()}.csv"
    resumen.to_csv(nombre_csv, index=False)
    print(f"Tabla completa guardada en: {nombre_csv}")

    return resumen


resumen_central = simular(theta_rmv, pval_rmv, "punto_central", conservador=False)
resumen_conservador = simular(theta_rmv, pval_rmv, "conservador_solo_significativos", conservador=True)


# ------------------------------------------------------------------
# 3. Guardar tabla a nivel de hogar afectado (opcional, para inspeccion)
# ------------------------------------------------------------------
columnas_guardar = ["ingreso_actual", "ingreso_nuevo", "delta_ln_ingreso", "X"] + [f"w_{g}" for g in grupos]
afectados[columnas_guardar].to_csv("hogares_afectados_rmv.csv", index=False)
print(f"\nDatos de los {len(afectados):,} hogares afectados guardados en: hogares_afectados_rmv.csv")

print(f"\n{'=' * 70}\nRESUMEN FINAL\n{'=' * 70}")
print(f"- {len(afectados):,} hogares con jefe/a empleado/obrero e ingreso en "
      f"[S/ {RMV_VIEJA:.0f}, S/ {RMV_NUEVA:.0f}) serian 'afectados' por el aumento de RMV.")
print(f"- Esta simulacion muestra SOLO la redistribucion de la canasta de alimentos")
print(f"  (gasto total en alimentos X se asume constante). El efecto de que X tambien")
print(f"  podria crecer con el ingreso (curva de Engel) NO esta incluido -- fue excluido")
print(f"  a pedido explicito del usuario.")
print(f"- Se reportan dos versiones: 'punto_central' (todos los theta_i tal cual estimados)")
print(f"  y 'conservador_solo_significativos' (theta_i no significativos al 5% puestos en 0).")

Hogares con datos completos (incluyendo ingreso del jefe/a, empleado/obrero): 7,508 de 33,418

Estimando theta_i -- muestra: ingreso_bajo_RMV (<= 1130) (2,064 hogares)

Estimando theta_i -- muestra: muestra_completa (referencia) (7,508 hogares)

Comparacion theta_i: submuestra <=RMV (usada en la simulacion) vs. muestra completa (referencia)
Grupo                   theta (<=RMV)   p-valor   theta (completa)   p-valor
------------------------------------------------------------------------------
aceites_grasas                -0.0026     0.2992            -0.0039*    0.0056
azucar_dulces                 +0.0022     0.3719            -0.0030*    0.0055
bebidas_alcoholicas           -0.0009     0.0790            -0.0001     0.4944
bebidas_no_alcoholicas        +0.0042     0.0997            +0.0058*    0.0000
cafe_te_infusiones            -0.0027     0.4287            -0.0092     0.6460
carnes_rojas                  +0.0056     0.2991            +0.0099*    0.0000
comida_fuera_hogar         

## Apendice: diagnosticos opcionales (no se corren automaticamente)

Estas celdas ya cumplieron su proposito durante el desarrollo del
modelo (llevaron a decisiones que ya estan incorporadas en la Parte 1
y 2: por ejemplo, usar solo el precio propio como endogeno por
ecuacion, o subdividir pan/pollo). Se dejan aqui documentadas por si
necesitas volver a investigar un problema similar (p.ej. si agregas
nuevos grupos y vuelven a aparecer signos "raros"), pero **no hace
falta correrlas** para reproducir `base_laids_final.parquet` ni los
resultados de la Parte 2.

Para correr cualquiera de estas, primero verifica que los archivos que
usan como entrada existan (algunas usan nombres de archivo de
versiones intermedias de la base que ya no se generan por defecto en
este notebook reestructurado).

### A.1 -- Verificar etiquetas de variables del modulo 601 (INEI)

In [18]:
"""
verificar_etiquetas.py
------------------------
Extrae las ETIQUETAS DE VARIABLE reales (puestas por el INEI) para
las columnas del modulo 601, sin cargar ninguna fila de datos.
Esto te dice con certeza que representa cada p601... antes de renombrar.
"""

import pandas as pd

ARCHIVO_DTA = "enaho01-2025-601.dta"

reader = pd.io.stata.StataReader(ARCHIVO_DTA)
etiquetas = reader.variable_labels()  # dict: nombre_columna -> descripcion

columnas_601 = [c for c in etiquetas if c.lower().startswith(("p601", "i601", "d601", "t601"))]

print("=== ETIQUETAS DE VARIABLES DEL MODULO 601 ===\n")
for col in sorted(columnas_601):
    print(f"{col:12s} -> {etiquetas[col]}")

=== ETIQUETAS DE VARIABLES DEL MODULO 601 ===

d601c        -> (Deflactado, anualizado) Cuanto fue el monto total de la compra
i601b2       -> (Imputado, anualizado) Cantidad de compra en kilo
i601c        -> (Imputado, anualizado) Monto de la compra
i601d2       -> (Imputado, anualizado) Cantidad obtenida en kilo
i601e        -> (Imputado, anualizado) Monto estimado
p601a        -> Codigo del producto
p601a1       -> Como obtuvieron el(la)...: Comprado
p601a2       -> Como obtuvieron el(la)...: Autoconsumo
p601a3       -> Como obtuvieron el(la)...: Autosuministro
p601a4       -> Como obtuvieron el(la)...: Como parte de pago a un miembro del hogar
p601a5       -> Como obtuvieron el(la)...: Regalado o pagado por algun miembro de otro hogar
p601a6       -> Como obtuvieron el(la)...: Regalado o donado por algun programa social
p601a7       -> Como obtuvieron el(la)...: Otro
p601b        -> En los ultimos 15 dias, del ... al... Ud. y/o algun miembro de este hogar obtuvieron, consumieron, c

### A.2 -- Diagnostico de fuerza de instrumentos (F-test de primera etapa)

Diagnostico que confirmo que usar los 14-17 instrumentos completos por
ecuacion generaba multicolinealidad severa entre instrumentos; la
conclusion (usar solo el instrumento propio z_i por ecuacion, sistema
exactamente identificado) ya esta incorporada en `estimar_laids_final.py`
(Parte 2.1). Requiere `base_laids_ancha.parquet` (version intermedia,
anterior al ajuste de calidad de Deaton) -- no se genera por defecto en
este notebook; si la necesitas, corre una version simplificada del Paso 9
sin el ajuste de calidad, o adapta las rutas a `base_laids_final.parquet`.

In [21]:
"""
diagnostico_instrumentos.py
------------------------------
Diagnostica la fuerza de los instrumentos leave-one-out:

  1. Para cada precio endogeno ln_precio_k, corre la regresion de
     primera etapa:  ln_precio_k ~ const + ln_X_sobre_P + z_1...z_14
     y calcula el F-test de significancia conjunta de los instrumentos
     (regla de Staiger-Stock: F > 10 = instrumento fuerte).

  2. Reporta la matriz de correlacion entre los instrumentos z_i,
     para ver si estan muy correlacionados entre si (lo cual
     explicaria el R^2 negativo tan extremo del sistema).
"""

import pandas as pd
import numpy as np
import statsmodels.api as sm
import pyarrow.parquet as pq  # <-- agregar

ARCHIVO = "base_laids_ancha.parquet"
df = pq.read_table(ARCHIVO).to_pandas()  # <-- en vez de pd.read_parquet(ARCHIVO)

grupos = sorted(c.replace("w_", "") for c in df.columns if c.startswith("w_"))
columnas_precio = [f"ln_precio_{g}" for g in grupos]
columnas_instrumento = [f"z_{g}" for g in grupos]

columnas_necesarias = columnas_precio + columnas_instrumento + ["ln_X_sobre_P"]
df_completo = df.dropna(subset=columnas_necesarias).copy()
print(f"Hogares usados en el diagnostico: {len(df_completo):,}\n")

# ============================================================
# 1. F-test de primera etapa por cada precio endogeno
# ============================================================
print("=== FUERZA DE LOS INSTRUMENTOS (F-test de primera etapa) ===")
print(f"{'Precio (endogeno)':30s} {'F-stat (instrumentos)':>22s} {'R2':>8s}  Interpretacion")
print("-" * 90)

resultados_f = []

for g in grupos:
    y = df_completo[f"ln_precio_{g}"]

    X_completo = sm.add_constant(df_completo[["ln_X_sobre_P"] + columnas_instrumento])
    modelo_completo = sm.OLS(y, X_completo).fit()

    X_restringido = sm.add_constant(df_completo[["ln_X_sobre_P"]])
    modelo_restringido = sm.OLS(y, X_restringido).fit()

    # F-test de significancia conjunta de los instrumentos (comparando modelos anidados)
    from statsmodels.stats.anova import anova_lm
    n = len(df_completo)
    k_completo = X_completo.shape[1]
    k_restringido = X_restringido.shape[1]
    rss_completo = modelo_completo.ssr
    rss_restringido = modelo_restringido.ssr
    df_num = k_completo - k_restringido
    df_den = n - k_completo
    f_stat = ((rss_restringido - rss_completo) / df_num) / (rss_completo / df_den)

    interpretacion = "FUERTE (F>10)" if f_stat > 10 else "DEBIL (F<10)"
    print(f"{g:30s} {f_stat:22.2f} {modelo_completo.rsquared:8.4f}  {interpretacion}")

    resultados_f.append({"grupo": g, "f_stat": f_stat, "r2": modelo_completo.rsquared})

tabla_f = pd.DataFrame(resultados_f)
tabla_f.to_csv("diagnostico_fuerza_instrumentos.csv", index=False)

n_debiles = (tabla_f["f_stat"] <= 10).sum()
print(f"\nInstrumentos debiles (F<=10): {n_debiles} de {len(grupos)}")

# ============================================================
# 2. Correlacion entre los instrumentos (posible multicolinealidad)
# ============================================================
print("\n=== CORRELACION ENTRE INSTRUMENTOS z_i (diagnostico de multicolinealidad) ===")

corr = df_completo[columnas_instrumento].corr()
corr_sin_diagonal = corr.where(~np.eye(len(corr), dtype=bool))

correlacion_promedio = corr_sin_diagonal.abs().mean().mean()
correlacion_maxima = corr_sin_diagonal.abs().max().max()

print(f"Correlacion absoluta promedio entre pares de instrumentos: {correlacion_promedio:.3f}")
print(f"Correlacion absoluta maxima entre pares de instrumentos:   {correlacion_maxima:.3f}")

if correlacion_promedio > 0.5:
    print("\n[DIAGNOSTICO] Los instrumentos estan MUY correlacionados entre si.")
    print("Esto es consistente con el R2 negativo extremo del sistema: al usar los")
    print("14 instrumentos (precios promedio de mercado de TODOS los bienes) para")
    print("instrumentar CADA precio, el sistema no logra separar variacion especifica")
    print("de cada bien -- probablemente porque el nivel general de precios sube y baja")
    print("junto (inflacion/deflacion regional-estacional general).")
    print("\nRECOMENDACION: usa solo el instrumento PROPIO de cada precio (z_i para")
    print("ln_precio_i), no los 14 instrumentos para cada ecuacion. Esto es lo estandar")
    print("en la practica (instrumento = 1 por variable endogena, sistema exactamente")
    print("identificado), y evita este problema de 'muchos instrumentos debiles'.")

corr.to_csv("correlacion_instrumentos.csv")
print("\nMatriz de correlacion completa guardada en: correlacion_instrumentos.csv")

Hogares usados en el diagnostico: 27,164

=== FUERZA DE LOS INSTRUMENTOS (F-test de primera etapa) ===
Precio (endogeno)               F-stat (instrumentos)       R2  Interpretacion
------------------------------------------------------------------------------------------
aceites_grasas                                  23.99   0.0147  FUERTE (F>10)
azucar_dulces                                   35.85   0.0799  FUERTE (F>10)
bebidas_alcoholicas                          55837.37   0.9691  FUERTE (F>10)
bebidas_no_alcoholicas                         191.40   0.1070  FUERTE (F>10)
cafe_te_infusiones                              12.13   0.0693  FUERTE (F>10)
carnes_rojas                                   118.18   0.2987  FUERTE (F>10)
comida_fuera_hogar                             154.51   0.1273  FUERTE (F>10)
frutas                                         103.81   0.2039  FUERTE (F>10)
frutos_secos_semillas                         3302.99   0.6657  FUERTE (F>10)
lacteos_huevos           

### A.3 -- Diagnostico de signos "raros" en pan_pasteleria_cereales / pollo_aves

Diagnostico que identifico sesgo de calidad (correlacion entre precio
pagado y gasto total del hogar) en estas dos categorias, lo que motivo
subdividirlas (Paso 6). Requiere `base_laids_ancha.parquet` (ver nota
en A.2).

In [22]:
"""
diagnostico_signos_raros.py
-------------------------------
Investiga por que 3 categorias (pan_pasteleria_cereales, pollo_aves,
y verduras_hortalizas implicito) salieron con gamma_ii de signo
"raro" (positivo, cuando la teoria de demanda espera negativo).

Dos chequeos:
  1. DISPERSION DE PRECIOS dentro de cada grupo: si el grupo mezcla
     productos muy distintos en precio (ej. pan corriente vs pasteles
     finos), el "precio promedio" agregado no es un precio de mercado
     limpio -- mezcla composicion/calidad, lo que puede invertir el
     signo esperado (sesgo de calidad, Deaton 1988).
  2. CORRELACION entre el precio usado y el gasto total del hogar (X):
     si los hogares con mayor gasto total tienden a pagar precios
     unitarios mas altos dentro del grupo, es la señal clasica de
     que el "precio" captura eleccion de calidad, no solo costo.

Compara los 3 grupos sospechosos contra 2 grupos "bien portados"
(pescados_mariscos, carnes_rojas) como referencia.
"""

import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa

ARCHIVO_ANCHA = "base_laids_ancha.parquet"
ARCHIVO_CRUDO = "enaho_601_con_wi.parquet"
ARCHIVO_MAPEO = "mapeo_grupo_aids.csv"

GRUPOS_SOSPECHOSOS = ["pan_pasteleria_cereales", "pollo_aves", "verduras_hortalizas"]
GRUPOS_REFERENCIA = ["pescados_mariscos", "carnes_rojas"]
GRUPOS_A_REVISAR = GRUPOS_SOSPECHOSOS + GRUPOS_REFERENCIA

# ============================================================
# 1. Dispersion de precios DENTRO de cada grupo (a nivel de PRODUCTO)
# ============================================================
print("=== 1. DISPERSION DE PRECIOS DENTRO DE CADA GRUPO (por producto) ===\n")

mapeo = pd.read_csv(ARCHIVO_MAPEO)
mapeo["codigo"] = mapeo["codigo"].astype(int)
mapa_grupo = dict(zip(mapeo["codigo"], mapeo["grupo_aids"]))

archivo_pq = pq.ParquetFile(ARCHIVO_CRUDO)
suma_gasto_prod = {}
suma_cantidad_prod = {}

for lote in archivo_pq.iter_batches(batch_size=100_000, columns=["codigo_producto", "gasto", "cantidad_comprada"]):
    df = pa.Table.from_batches([lote]).to_pandas()
    df["grupo"] = df["codigo_producto"].map(mapa_grupo)
    df = df[df["grupo"].isin(GRUPOS_A_REVISAR)]
    df = df.dropna(subset=["gasto", "cantidad_comprada"])
    df = df[df["cantidad_comprada"] > 0]

    agr = df.groupby(["grupo", "codigo_producto"]).agg(
        gasto=("gasto", "sum"), cantidad=("cantidad_comprada", "sum")
    )
    for (grupo, cod), fila in agr.iterrows():
        clave = (grupo, cod)
        suma_gasto_prod[clave] = suma_gasto_prod.get(clave, 0) + fila["gasto"]
        suma_cantidad_prod[clave] = suma_cantidad_prod.get(clave, 0) + fila["cantidad"]

filas = []
for (grupo, cod), gasto in suma_gasto_prod.items():
    cantidad = suma_cantidad_prod.get((grupo, cod), np.nan)
    if cantidad and cantidad > 0:
        filas.append({"grupo": grupo, "codigo": cod, "precio_promedio_producto": gasto / cantidad})

precios_producto = pd.DataFrame(filas)

print(f"{'Grupo':30s} {'N productos':>12s} {'Precio min':>12s} {'Precio max':>12s} {'CV (disp.)':>12s}")
print("-" * 82)
for grupo in GRUPOS_A_REVISAR:
    sub = precios_producto[precios_producto["grupo"] == grupo]["precio_promedio_producto"]
    sub = sub[(sub > 0) & np.isfinite(sub)]
    if len(sub) > 1:
        cv = sub.std() / sub.mean()
        print(f"{grupo:30s} {len(sub):12d} {sub.min():12.2f} {sub.max():12.2f} {cv:12.2f}")
    else:
        print(f"{grupo:30s} {len(sub):12d}  (insuficientes datos)")

print("\n(CV = coeficiente de variacion = desv.estandar / media. Mientras mas alto,")
print(" mas heterogeneo es el grupo -- mezcla productos muy baratos y muy caros)")

# ============================================================
# 2. Correlacion entre precio usado y gasto total del hogar (X)
# ============================================================
print("\n\n=== 2. CORRELACION entre precio usado (ln) y gasto total del hogar (ln X) ===\n")

df_ancha = pq.read_table(ARCHIVO_ANCHA).to_pandas()  # <-- en vez de pd.read_parquet(ARCHIVO_ANCHA)
df_ancha["ln_X"] = np.log(df_ancha["X"])

print(f"{'Grupo':30s} {'Correlacion (solo compradores)':>32s} {'% hogares que compraron':>26s}")
print("-" * 92)
for grupo in GRUPOS_A_REVISAR:
    col_w = f"w_{grupo}"
    col_precio = f"ln_precio_{grupo}"
    compradores = df_ancha[df_ancha[col_w] > 0]
    corr = compradores[col_precio].corr(compradores["ln_X"])
    pct_compradores = 100 * len(compradores) / len(df_ancha)
    print(f"{grupo:30s} {corr:32.3f} {pct_compradores:26.1f}")

print("\n(Correlacion positiva y alta = hogares con mas gasto total pagan precios")
print(" unitarios mas altos dentro del grupo -- señal de eleccion de calidad,")
print(" no de un precio de mercado 'limpio'. Esto sesga gamma_ii hacia positivo.)")

=== 1. DISPERSION DE PRECIOS DENTRO DE CADA GRUPO (por producto) ===

Grupo                           N productos   Precio min   Precio max   CV (disp.)
----------------------------------------------------------------------------------
pan_pasteleria_cereales                 113         0.01        19.87         4.08
pollo_aves                               18         0.02        21.07         2.59
verduras_hortalizas                     145         0.01        12.50         3.79
pescados_mariscos                        97         0.01        77.94         3.30
carnes_rojas                             94         0.01        35.00         2.44

(CV = coeficiente de variacion = desv.estandar / media. Mientras mas alto,
 mas heterogeneo es el grupo -- mezcla productos muy baratos y muy caros)


=== 2. CORRELACION entre precio usado (ln) y gasto total del hogar (ln X) ===

Grupo                            Correlacion (solo compradores)    % hogares que compraron
---------------------------